# Quantum Repeater Digital Twin -- Refactor v3

**Fixes applied in this version relative to v2:**

1. **Statistical fragility in the Pareto Frontier (fixed, not just
   documented).** v2 trained the `EdgeLSTM` with a single batch and a
   single seed per `lambda_penalty` value, and only *acknowledged* in
   markdown that this left individual points vulnerable to local optima.
   Here, `run_pareto_sweep` trains and evaluates **each lambda on 5
   independent seeds** (configurable), and every row of the final table
   reports **mean ± standard deviation** per metric -- making training
   variance visible instead of hidden, which is essential before using any
   point on the curve to decide the production admission controller.
2. **Hardware-level micro-profiling.** `time.perf_counter()` measures
   host-side wall-clock time and is susceptible to OS jitter even with
   `torch.cuda.synchronize()`. Inference latency is now measured with
   `torch.cuda.Event(enable_timing=True)` on GPU -- hardware markers
   directly on the CUDA stream -- with `perf_counter()` kept only as a
   *fallback* on CPU (where there is no asynchronous stream to instrument).
3. **Decomposition into a reusable package.** The entire pipeline (channel
   simulator, model, quantum dataplane, orchestrator, Pareto sweep, CLI)
   has been extracted from the notebook into the `qrepeater_twin/` package,
   testable with `pytest` and runnable outside any Jupyter kernel
   (`python -m qrepeater_twin.cli`). This notebook is now just a thin
   driver: the cells below materialize the package on disk (via
   `%%writefile`, so the notebook stays self-contained and reproducible in
   Colab) and then import and run it -- business logic no longer lives in
   loose cells.

PyTorch device handling (CPU/GPU) and the Qiskit noise-model injection
(thermal relaxation channel $T_2$ penalized by the isolated classical
latency) remain unchanged.

**New in this revision -- baseline comparison (Section 6):**

4. **Cross-architecture predictor comparison.** Beyond sweeping
   `lambda_penalty` for a fixed `EdgeLSTM + CS_MSELoss`, the pipeline now
   trains and evaluates four alternative fidelity predictors through the
   SAME digital-twin loop: **LSTM+MSE** (same architecture, plain
   cost-insensitive loss), **Random Forest**, **XGBoost** (optional
   dependency, skipped with a warning if absent), and a small
   **Transformer** encoder (`qrepeater_twin/baselines.py`).
5. **Operational metrics for deployment decisions.** `qrepeater_twin/metrics.py`
   adds **throughput** (useful pairs/s), **QPU economy** (purification
   attempts/shots avoided vs. blind purification), and an illustrative
   **energy** accounting (quantum gate/shot + classical inference energy),
   on top of the QPU-yield/latency metrics already reported by the Pareto
   sweep.
6. **Ranked decision matrix.** `build_decision_matrix` normalizes every
   weighted criterion (direction-aware) across all candidate models and
   computes a weighted composite score, so Section 6 ends with a single
   ranked table (`Rank 1` = recommended model) instead of several
   metrics the reader has to reconcile by eye.


## 0. Dependency installation (Google Colab)

In [ ]:
# Qiskit and Qiskit Aer are usually not pre-installed on Colab.
# torch, numpy, pandas and scikit-learn are already available by default in the Colab runtime.
!pip install -q "qiskit>=1.0" "qiskit-aer>=0.14" --upgrade


## 1. Materializing the `qrepeater_twin/` package

The cells below write the decomposed package to disk (one module per
responsibility). In a real GitHub repository, these files would already
exist as versioned `.py` files and these `%%writefile` cells would be
unnecessary -- they exist here only so the notebook remains self-contained
and runnable end-to-end on a clean Colab runtime, without depending on
`git clone`.


In [ ]:
import os
os.makedirs("qrepeater_twin", exist_ok=True)
os.makedirs("tests", exist_ok=True)


### 1.1 `qrepeater_twin/__init__.py` -- public package API

In [ ]:
%%writefile qrepeater_twin/__init__.py
"""
qrepeater_twin
==============

Digital Twin of a Quantum Repeater with a predictive admission controller
(EdgeLSTM) and a Pareto Frontier sweep over the False Positive penalty of
the CS_MSELoss -- plus a cross-architecture comparison against LSTM+MSE,
Random Forest, XGBoost, and Transformer baselines, with throughput, QPU
economy, and energy accounting, and a ranked multi-criteria decision
matrix.

This package decomposes the original prototype (a single monolithic
notebook) into independent, testable modules, suitable for a reproducible
GitHub repository:

    channel_simulator.py  -> WDMChannelSimulator (synthetic data generation)
    models.py               -> EdgeLSTM, CS_MSELoss, train_edge_lstm
    timing.py                 -> InferenceTimer (hardware-accurate profiling, CUDA Events)
    quantum_node.py            -> QuantumRepeaterNode (virtual quantum dataplane via Qiskit Aer)
    orchestrator.py             -> DigitalTwinOrchestrator (intelligent/blind loops)
    pareto_sweep.py               -> run_pareto_sweep (statistically robust, multi-seed)
    baselines.py                   -> LSTM+MSE, Random Forest, XGBoost, Transformer predictors
    metrics.py                      -> throughput, QPU economy, energy, decision matrix
    model_comparison.py              -> run_model_comparison (cross-architecture, multi-seed)
    config.py                         -> configuration dataclasses
    cli.py                              -> command-line entry point (main)
"""

from .channel_simulator import WDMChannelSimulator
from .models import EdgeLSTM, CS_MSELoss, train_edge_lstm
from .timing import InferenceTimer
from .quantum_node import QuantumRepeaterNode
from .orchestrator import DigitalTwinOrchestrator
from .pareto_sweep import run_pareto_sweep
from .baselines import (
    TinyTransformer,
    RandomForestFidelityModel,
    XGBoostFidelityModel,
    train_lstm_mse,
    train_random_forest,
    train_xgboost,
    train_transformer,
)
from .metrics import (
    compute_throughput,
    compute_qpu_economy,
    compute_energy_report,
    build_decision_matrix,
)
from .model_comparison import run_model_comparison
from .config import (
    SimConfig,
    TrainConfig,
    QuantumConfig,
    SweepConfig,
    BaselineConfig,
    EnergyConfig,
    ComparisonConfig,
)

__all__ = [
    "WDMChannelSimulator",
    "EdgeLSTM",
    "CS_MSELoss",
    "train_edge_lstm",
    "InferenceTimer",
    "QuantumRepeaterNode",
    "DigitalTwinOrchestrator",
    "run_pareto_sweep",
    "TinyTransformer",
    "RandomForestFidelityModel",
    "XGBoostFidelityModel",
    "train_lstm_mse",
    "train_random_forest",
    "train_xgboost",
    "train_transformer",
    "compute_throughput",
    "compute_qpu_economy",
    "compute_energy_report",
    "build_decision_matrix",
    "run_model_comparison",
    "SimConfig",
    "TrainConfig",
    "QuantumConfig",
    "SweepConfig",
    "BaselineConfig",
    "EnergyConfig",
    "ComparisonConfig",
]

__version__ = "3.0.0"


### 1.2 `qrepeater_twin/config.py` -- configuration dataclasses (Sim/Train/Quantum/Sweep)

In [ ]:
%%writefile qrepeater_twin/config.py
"""
Configuration dataclasses.

Centralizing hyperparameters here (instead of scattering them as
positional/keyword arguments across several functions, as in the original
notebook) improves reproducibility: an entire `SweepConfig` can be
serialized (JSON/YAML), version-controlled, and cited in a README or an
experiment report.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import List


@dataclass
class SimConfig:
    """Configuration for the synthetic data generator (WDMChannelSimulator)."""

    n_steps: int = 4000
    dt: float = 0.01
    seed: int = 42
    window_size: int = 20
    test_size: float = 0.2


@dataclass
class TrainConfig:
    """Training configuration for EdgeLSTM + CS_MSELoss."""

    hidden_size: int = 16
    epochs: int = 150
    lr: float = 0.012
    threshold: float = 0.65
    lambda_fn: float = 4.0
    discard_penalty_weight: float = 10.0
    max_discard_rate: float = 0.60


@dataclass
class QuantumConfig:
    """Configuration for the virtual quantum dataplane (QuantumRepeaterNode)."""

    T1: float = 50e-6
    T2: float = 30e-6
    depol_prob: float = 0.01
    shots: int = 512
    seed: int = 7
    success_rate_cutoff: float = 0.5


@dataclass
class SweepConfig:
    """
    Configuration for the Pareto Frontier sweep over `lambda_penalty`.

    `seeds` fixes the statistical fragility of the original prototype:
    instead of a single training run (single batch + single seed) per
    lambda value, each point on the Pareto Frontier is the mean (± standard
    deviation) of `len(seeds)` independent training runs, reducing the risk
    of any given point getting stuck in an unrepresentative local optimum.
    """

    lambda_values: List[float] = field(default_factory=lambda: [1.0, 2.0, 5.0, 10.0, 20.0, 50.0])
    seeds: List[int] = field(default_factory=lambda: [42, 43, 44, 45, 46])


@dataclass
class BaselineConfig:
    """
    Hyperparameters for the *predictor* baselines compared against the
    intelligent admission controller (EdgeLSTM + CS_MSELoss):

        - LSTM+MSE       : same EdgeLSTM architecture, trained with a plain
                            (cost-insensitive) nn.MSELoss -- isolates the
                            contribution of CS_MSELoss itself, holding the
                            architecture fixed.
        - Random Forest   : classical, non-recurrent regressor over the
                            flattened window (n_estimators/max_depth below).
        - XGBoost         : gradient-boosted trees over the flattened
                            window (n_estimators/max_depth/learning_rate
                            below). Optional dependency -- skipped with a
                            warning (not a hard failure) if `xgboost` is
                            not installed.
        - Transformer     : small Transformer encoder over the same input
                            window, trained with plain nn.MSELoss, as a
                            higher-capacity architectural baseline.
    """

    # LSTM + plain MSE (same architecture as EdgeLSTM, no CS_MSELoss)
    lstm_mse_hidden_size: int = 16
    lstm_mse_epochs: int = 150
    lstm_mse_lr: float = 0.012

    # Random Forest
    rf_n_estimators: int = 200
    rf_max_depth: int = 8

    # XGBoost
    xgb_n_estimators: int = 200
    xgb_max_depth: int = 5
    xgb_learning_rate: float = 0.1

    # Transformer encoder
    transformer_d_model: int = 32
    transformer_nhead: int = 4
    transformer_num_layers: int = 2
    transformer_dim_feedforward: int = 64
    transformer_epochs: int = 150
    transformer_lr: float = 0.005


@dataclass
class EnergyConfig:
    """
    Coefficients of the (illustrative) energy-accounting model used by
    `qrepeater_twin.metrics.compute_energy_report`.

    These are order-of-magnitude, documented estimates -- not vendor
    datasheet values -- meant to make the *relative* energy trade-off
    between "ask the network first" (predictive admission, pays a small,
    constant classical-inference energy on every cycle) and "always
    purify" (blind baseline, pays the full quantum-operation energy on
    every cycle) visible and comparable across predictor models.

        - joules_per_1q_gate / joules_per_2q_gate : energy per logical
          gate operation executed by the QPU (or its control electronics)
          during one BBPSSW purification attempt.
        - joules_per_shot_overhead                : fixed per-shot
          overhead (state prep + measurement + reset) independent of gate
          count.
        - classical_inference_power_w             : average power draw of
          the edge accelerator while a predictor model runs one forward
          pass / one prediction (Watts). Combined with the measured
          per-cycle latency to obtain per-cycle classical energy.
        - classical_idle_power_w                  : power draw of that
          same edge device when it is *not* running a predictor at all
          (the blind baseline never invokes one), included so the blind
          baseline's classical energy isn't silently zero.
    """

    joules_per_1q_gate: float = 5e-9
    joules_per_2q_gate: float = 2e-8
    joules_per_shot_overhead: float = 1e-9
    classical_inference_power_w: float = 0.5
    classical_idle_power_w: float = 0.05

    # BBPSSW circuit gate counts (see quantum_node.build_bbpssw_circuit):
    # 2x H, 2x CX (Bell-pair prep) + 4x id + 2x CX (bilateral CNOTs).
    gates_1q_per_attempt: int = 6
    gates_2q_per_attempt: int = 4


@dataclass
class ComparisonConfig:
    """
    Configuration for `qrepeater_twin.model_comparison.run_model_comparison`,
    which trains/evaluates every predictor baseline (EdgeLSTM+CS_MSELoss at
    a representative lambda, LSTM+MSE, Random Forest, XGBoost, Transformer)
    under the same multi-seed protocol as `run_pareto_sweep`, and reports
    throughput, QPU economy, energy, and a multi-criteria decision matrix.
    """

    representative_lambda: float = 10.0
    seeds: List[int] = field(default_factory=lambda: [42, 43, 44, 45, 46])
    include_xgboost: bool = True
    cycle_time_s: float = 1e-3
    # Weights for the decision matrix (must be non-negative; renormalized
    # internally). Each criterion is oriented so that a HIGHER weighted
    # score is always better (cost-type criteria, e.g. latency/energy, are
    # inverted before weighting).
    decision_weights: dict = field(default_factory=lambda: {
        "qpu_yield_pct": 0.25,
        "throughput_pairs_per_s": 0.20,
        "qpu_cycles_saved_pct": 0.20,
        "energy_saved_pct": 0.20,
        "inference_latency_ms": 0.15,
    })


### 1.3 `qrepeater_twin/channel_simulator.py` -- `WDMChannelSimulator`

Generates synthetic time series via an Ornstein-Uhlenbeck process and
derives the latent quantum fidelity $F(t)$. Logic identical to v2; only
extracted into an importable module that can be tested in isolation.

In [ ]:
%%writefile qrepeater_twin/channel_simulator.py
"""
Component 1 -- `WDMChannelSimulator`.

Generates synthetic time series via an Ornstein-Uhlenbeck process and
derives the latent quantum fidelity F(t). Logic identical to the original
prototype; only extracted into an importable, testable module.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler


class WDMChannelSimulator:
    """
    Synthetic simulator of a WDM (Wavelength Division Multiplexing) optical
    channel operating at the network edge.

    Generates two continuous physical variables via Ornstein-Uhlenbeck (OU)
    processes:

        - phase_deviation : classical optical signal phase deviation (rad, >= 0)
        - temp_gradient   : local temperature gradient (K/m, >= 0)

    The latent quantum fidelity F(t) is derived from the phase deviation:
    inversely proportional to it, plus Gaussian noise, clipped to [0, 1].
    """

    def __init__(self, n_steps: int = 4000, dt: float = 0.01, seed: int = 42):
        self.n_steps = n_steps
        self.dt = dt
        self.rng = np.random.default_rng(seed)

    def _ornstein_uhlenbeck(self, theta: float, mu: float, sigma: float, x0: float) -> np.ndarray:
        """
        Numerically integrates (Euler-Maruyama) an Ornstein-Uhlenbeck process:
            dX_t = theta * (mu - X_t) * dt + sigma * dW_t
        """
        x = np.zeros(self.n_steps, dtype=np.float64)
        x[0] = x0
        sqrt_dt = np.sqrt(self.dt)
        for t in range(1, self.n_steps):
            dW = self.rng.normal(0.0, sqrt_dt)
            x[t] = x[t - 1] + theta * (mu - x[t - 1]) * self.dt + sigma * dW
        return x

    def generate_dataset(self) -> pd.DataFrame:
        """Generates the full synthetic dataset (features + latent fidelity)."""
        phase_deviation = self._ornstein_uhlenbeck(theta=0.70, mu=0.30, sigma=0.15, x0=0.30)
        phase_deviation = np.abs(phase_deviation)

        temp_gradient = self._ornstein_uhlenbeck(theta=0.50, mu=0.50, sigma=0.10, x0=0.50)
        temp_gradient = np.abs(temp_gradient)

        alpha = 1.4
        eps = self.rng.normal(0.0, 0.03, self.n_steps)
        fidelity = 1.0 - alpha * phase_deviation + eps
        fidelity = np.clip(fidelity, 0.0, 1.0)

        return pd.DataFrame({
            "phase_deviation": phase_deviation,
            "temp_gradient": temp_gradient,
            "fidelity": fidelity,
        })

    def preprocess(self, df: pd.DataFrame, window_size: int = 20, test_size: float = 0.2):
        """
        Normalizes features with MinMaxScaler, builds sliding windows
        (batch, seq_len, n_features), and splits train/test without
        shuffling (preserves chronological order).
        """
        features = df[["phase_deviation", "temp_gradient"]].values
        target = df[["fidelity"]].values

        feat_scaler = MinMaxScaler(feature_range=(0.0, 1.0))
        features_scaled = feat_scaler.fit_transform(features)

        X, y = [], []
        for i in range(len(features_scaled) - window_size):
            X.append(features_scaled[i:i + window_size])
            y.append(target[i + window_size])
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32)

        split_idx = int(len(X) * (1.0 - test_size))
        X_train, X_test = X[:split_idx], X[split_idx:]
        y_train, y_test = y[:split_idx], y[split_idx:]

        X_train_t = torch.tensor(X_train, dtype=torch.float32)
        y_train_t = torch.tensor(y_train, dtype=torch.float32)
        X_test_t = torch.tensor(X_test, dtype=torch.float32)
        y_test_t = torch.tensor(y_test, dtype=torch.float32)

        return X_train_t, y_train_t, X_test_t, y_test_t, feat_scaler


### 1.4 `qrepeater_twin/timing.py` -- `InferenceTimer` (micro-profiling fix)

A context manager that times the forward pass using `torch.cuda.Event`
(hardware measurement, on the CUDA stream) when the device is a GPU, and
falls back to `time.perf_counter()` only on CPU. Isolating this logic in
its own module also makes it easier to test and reuse outside this
pipeline.

In [ ]:
%%writefile qrepeater_twin/timing.py
"""
High-precision profiling utility for PyTorch inference.

Fixes the "Micro-Profiling Inaccuracies" flaw from the original prototype:
`time.perf_counter()` measures host-side (CPU) wall-clock time. Even with
`torch.cuda.synchronize()` called before/after the forward pass -- which
guarantees *correctness* (the measurement won't stop before the CUDA kernel
has actually finished) -- the value itself still includes OS scheduler
jitter, Python/CUDA context-switch overhead, and the limited resolution of
the host clock. For inference in the microsecond/millisecond range on a
compact network like EdgeLSTM, that jitter can be on the same order of
magnitude as the signal being measured.

`torch.cuda.Event(enable_timing=True)` inserts markers directly into the
CUDA stream and measures elapsed time in hardware (on the GPU), via
`elapsed_time()`, isolating the measurement from host-side jitter. It is
the mechanism recommended by the PyTorch documentation for benchmarking
GPU inference latency.

CPU has no equivalent "hardware event" concept (there is no asynchronous
stream to synchronize against), so `time.perf_counter()` remains the best
available option in that case -- and is used only as a fallback.
"""

from __future__ import annotations

import time

import torch


class InferenceTimer:
    """
    Context manager that times a block of code (typically `model(x)`),
    automatically choosing the most accurate profiling mechanism available
    for the given `device`:

        - device.type == "cuda" -> torch.cuda.Event (hardware-level
          measurement, immune to host-side OS jitter).
        - otherwise (CPU)        -> time.perf_counter() (best-effort
          fallback; CPU exposes no asynchronous stream events).

    Usage:
        with InferenceTimer(device) as timer:
            pred = model(x)
        tau_inf = timer.elapsed_s  # seconds, always in this unit
    """

    def __init__(self, device: torch.device):
        self.device = device
        self.use_cuda_events = device.type == "cuda"
        self.elapsed_s: float = 0.0

        if self.use_cuda_events:
            self._start_evt = torch.cuda.Event(enable_timing=True)
            self._end_evt = torch.cuda.Event(enable_timing=True)
        else:
            self._t0 = 0.0

    def __enter__(self) -> "InferenceTimer":
        if self.use_cuda_events:
            # Drain the stream before marking the start: ensures no
            # previously pending work is included in the measured window.
            torch.cuda.synchronize()
            self._start_evt.record()
        else:
            self._t0 = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb) -> None:
        if self.use_cuda_events:
            self._end_evt.record()
            # Required: elapsed_time() requires both events to have
            # already completed on the device.
            torch.cuda.synchronize()
            elapsed_ms = self._start_evt.elapsed_time(self._end_evt)
            self.elapsed_s = elapsed_ms / 1000.0
        else:
            self.elapsed_s = time.perf_counter() - self._t0


### 1.5 `qrepeater_twin/models.py` -- `EdgeLSTM`, `CS_MSELoss`, `train_edge_lstm`

`CS_MSELoss` exposes `lambda_penalty` as the hyperparameter swept over the
Pareto Frontier. `train_edge_lstm` now explicitly accepts a `seed`, a
prerequisite for the multi-seed averaging in `pareto_sweep.py`.

In [ ]:
%%writefile qrepeater_twin/models.py
"""
Component 2 -- `EdgeLSTM` and `CS_MSELoss` (parameterized for the sweep).

`CS_MSELoss` exposes `lambda_penalty` as the main hyperparameter of the
Pareto Frontier, instantiable dynamically: `CS_MSELoss(lambda_penalty=L)`.
The remaining terms (a moderate False Negative penalty and excess-discard
regularization) remain as stabilizers that prevent the model from
trivially collapsing at any point of the lambda sweep.

`train_edge_lstm` explicitly accepts a `seed` and calls `torch.manual_seed`
before any parameter initialization -- a prerequisite for the multi-seed
averaging implemented in `pareto_sweep.py` (each seed must produce a
genuinely different, reproducible weight initialization).
"""

from __future__ import annotations

import torch
import torch.nn as nn


class EdgeLSTM(nn.Module):
    """
    Lightweight recurrent neural network ("Edge LSTM"), designed for fast
    inference on resource-constrained edge hardware.

    Architecture:
        input   -> (batch, seq_len, 2)  [phase_deviation, temp_gradient]
        LSTM    -> compact hidden_size, few layers
        output  -> linear layer + sigmoid, producing F_hat(t) in [0, 1]
    """

    def __init__(self, input_size: int = 2, hidden_size: int = 16, num_layers: int = 1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.head = nn.Linear(hidden_size, 1)
        self.activation = nn.Sigmoid()  # ensures F_hat(t) is in [0, 1]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]
        pred = self.activation(self.head(last_hidden))
        return pred


class CS_MSELoss(nn.Module):
    """
    Cost-Sensitive Mean Squared Error (CS-MSE), parameterized as the "knob"
    of the Pareto Frontier.

    Terms:
        - lambda_penalty : SEVERE penalty on False Positives (F_true < threshold
                            <= F_pred). This is the hyperparameter swept over
                            in the optimization loop -- the higher it is, the
                            more conservative the model, the higher the QPU
                            efficiency, and the lower the throughput.
        - lambda_fn       : MODERATE penalty on False Negatives (F_pred <
                            threshold <= F_true), kept fixed during the sweep
                            to avoid the model fully collapsing into
                            "discard everything" at high lambda_penalty
                            values.
        - discard_penalty_weight / max_discard_rate : batch-level
          regularization that penalizes discard rates above
          max_discard_rate, reinforcing training stability across the
          whole sweep.
    """

    def __init__(self, threshold: float = 0.65, lambda_penalty: float = 10.0,
                 lambda_fn: float = 2.0, discard_penalty_weight: float = 5.0,
                 max_discard_rate: float = 0.60):
        super().__init__()
        self.threshold = threshold
        self.lambda_penalty = lambda_penalty
        self.lambda_fn = lambda_fn
        self.discard_penalty_weight = discard_penalty_weight
        self.max_discard_rate = max_discard_rate

    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        squared_error = (y_pred - y_true) ** 2

        is_false_positive = (y_true < self.threshold) & (y_pred >= self.threshold)
        is_false_negative = (y_true >= self.threshold) & (y_pred < self.threshold)

        weights = torch.ones_like(squared_error)
        weights = torch.where(is_false_positive, torch.full_like(squared_error, self.lambda_penalty), weights)
        weights = torch.where(is_false_negative, torch.full_like(squared_error, self.lambda_fn), weights)

        weighted_mse = (squared_error * weights).mean()

        # Excess-discard penalty (batch level), differentiable via sigmoid.
        soft_discard_indicator = torch.sigmoid((self.threshold - y_pred) * 50.0)
        discard_rate = soft_discard_indicator.mean()
        excess_discard = torch.clamp(discard_rate - self.max_discard_rate, min=0.0)
        discard_penalty = self.discard_penalty_weight * (excess_discard ** 2)

        return weighted_mse + discard_penalty


def train_edge_lstm(model: nn.Module, X_train: torch.Tensor, y_train: torch.Tensor,
                     threshold: float = 0.65, lambda_penalty: float = 10.0, lambda_fn: float = 2.0,
                     discard_penalty_weight: float = 5.0, max_discard_rate: float = 0.60,
                     epochs: int = 120, lr: float = 3e-3, device: torch.device = None,
                     seed: int = None, verbose: bool = False):
    """
    Single-batch (full-batch) training routine (compact dataset).

    `device` determines where the model (and implicitly the tensors, which
    must already be on the same device) is trained. Kept explicit to allow
    consistent GPU/CPU handling throughout the pipeline.

    When provided, `seed` is re-applied via `torch.manual_seed` at the start
    of this function, for reproducibility of the optimization loop itself
    (Adam's internal operation order, etc.). **Important**: since training
    here is full-batch (no DataLoader/shuffling), the only real source of
    variation between seeds is the model's weight initialization -- and
    that initialization must already have happened *before* this call,
    with the same seed applied immediately before `EdgeLSTM(...)` is
    instantiated. This is exactly the pattern (seed -> build model -> train)
    that `pareto_sweep.run_pareto_sweep` follows on every round of the
    multi-seed averaging, ensuring each round starts from a genuinely
    different, reproducible initialization.
    """
    if seed is not None:
        torch.manual_seed(seed)

    if device is not None:
        model = model.to(device)

    criterion = CS_MSELoss(threshold=threshold, lambda_penalty=lambda_penalty, lambda_fn=lambda_fn,
                            discard_penalty_weight=discard_penalty_weight,
                            max_discard_rate=max_discard_rate)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()
        if verbose and (epoch + 1) % 30 == 0:
            with torch.no_grad():
                discard_rate_now = (y_pred < threshold).float().mean().item()
            print(f"    Epoch {epoch + 1:3d}/{epochs} | CS-MSE Loss: {loss.item():.6f} | "
                  f"Discard rate (train): {discard_rate_now*100:.1f}%")
    return model


### 1.6 `qrepeater_twin/quantum_node.py` -- `QuantumRepeaterNode`

Implements the BBPSSW protocol under NISQ noise (depolarization +
$T_1/T_2$), with the circuit built and transpiled exactly once per
instance.

In [ ]:
%%writefile qrepeater_twin/quantum_node.py
"""
Component 3 -- Virtual Quantum Dataplane (`QuantumRepeaterNode`).

Implements the BBPSSW protocol under NISQ noise (depolarization + T1/T2),
with the "logical latency clock" applied via a thermal-relaxation channel
e^(-latency/T2). The circuit is built and transpiled exactly once per
instance (avoids thousands of redundant recompilations during the Pareto
sweep, since the circuit structure doesn't change across runs -- only the
simulator's noise model does).
"""

from __future__ import annotations

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error


class QuantumRepeaterNode:
    """
    Virtual quantum dataplane of a repeater node, emulated via Qiskit Aer.

    Implements the BBPSSW entanglement-purification circuit under a custom
    NISQ noise model (depolarization + T1/T2 relaxation), and exposes a
    "logical latency clock" that ages the quantum memory in proportion to
    the isolated classical inference time.
    """

    def __init__(self, T1: float = 50e-6, T2: float = 30e-6,
                 depol_prob: float = 0.01, shots: int = 512, seed: int = 7):
        assert T2 <= 2 * T1, "Physical constraint: T2 must be <= 2*T1"
        self.T1 = T1
        self.T2 = T2
        self.depol_prob = depol_prob
        self.shots = shots
        self.seed = seed

        self.base_noise_model = self._build_noise_model()
        self.simulator = AerSimulator(noise_model=self.base_noise_model, seed_simulator=seed)

        # The BBPSSW circuit is structural and doesn't change across runs --
        # only the simulator's noise model varies (via apply_latency_decay).
        # For this reason it is built and transpiled ONCE here, and reused
        # by run_purification() on every subsequent call, eliminating the
        # bottleneck of thousands of recompilations in the simulation loop /
        # Pareto sweep.
        self._circuit = self.build_bbpssw_circuit()
        self._compiled_circuit = transpile(self._circuit, self.simulator)

    def _build_noise_model(self, extra_relax_error=None) -> NoiseModel:
        """
        Builds the NISQ noise model: depolarization + T1/T2 relaxation on
        logical gates, and (optionally) an additional relaxation error
        mapped to the 'id' gate, used to represent aging due to classical
        latency.
        """
        noise_model = NoiseModel()

        error_1q = depolarizing_error(self.depol_prob, 1)
        error_2q = depolarizing_error(self.depol_prob * 2, 2)

        gate_time_1q = 50e-9
        gate_time_2q = 300e-9

        thermal_1q = thermal_relaxation_error(self.T1, self.T2, gate_time_1q)
        thermal_2q_single = thermal_relaxation_error(self.T1, self.T2, gate_time_2q)
        thermal_2q = thermal_2q_single.tensor(thermal_2q_single)

        # Composes depolarization + thermal relaxation into a single
        # QuantumError per gate, avoiding multiple calls to
        # add_all_qubit_quantum_error on the same instruction (which would
        # generate redundant composition warnings).
        combined_1q = error_1q.compose(thermal_1q)
        combined_2q = error_2q.compose(thermal_2q)

        noise_model.add_all_qubit_quantum_error(combined_1q, ["u1", "u2", "u3", "x", "h"])
        noise_model.add_all_qubit_quantum_error(combined_2q, ["cx"])

        if extra_relax_error is not None:
            noise_model.add_all_qubit_quantum_error(extra_relax_error, ["id"])

        return noise_model

    def apply_latency_decay(self, latency: float) -> AerSimulator:
        """
        Logical latency clock.

        Takes the computational time (in seconds) measured in isolation
        (strictly the EdgeLSTM forward pass, or 0.0 in the blind baseline)
        and builds a simulator whose noise model includes a thermal
        relaxation channel equivalent to that interval, applied to the
        'id' gate of the circuit (quantum memory sitting idle, waiting on
        the classical AI's decision). With latency=0.0 (the baseline case),
        the aging channel is effectively null, reflecting unconditional
        admission with no additional wait.
        """
        latency = max(latency, 0.0)
        aging_error = thermal_relaxation_error(self.T1, self.T2, latency) if latency > 0.0 else None
        aged_noise_model = self._build_noise_model(extra_relax_error=aging_error)
        aged_simulator = AerSimulator(noise_model=aged_noise_model, seed_simulator=self.seed)
        return aged_simulator

    @staticmethod
    def build_bbpssw_circuit() -> QuantumCircuit:
        """
        BBPSSW entanglement-purification circuit.

        Qubits 0, 1 -> Bell pair "A"; qubits 2, 3 -> Bell pair "B" (sacrificed).
        1) Creates the two Bell pairs.
        2) 'id' gate on all qubits: represents the wait in quantum memory
           (the target of latency-driven aging).
        3) Bilateral CNOTs of the BBPSSW protocol.
        4) Measures the control qubits (sacrificed pair) in the Z basis;
           matching outcomes (00 or 11) indicate successful purification.
        """
        qc = QuantumCircuit(4, 2, name="BBPSSW")

        for a, b in [(0, 1), (2, 3)]:
            qc.h(a)
            qc.cx(a, b)
        qc.barrier()

        for q in range(4):
            qc.id(q)
        qc.barrier()

        qc.cx(0, 2)
        qc.cx(1, 3)
        qc.barrier()

        qc.measure(2, 0)
        qc.measure(3, 1)

        return qc

    def run_purification(self, simulator: AerSimulator = None):
        """
        Runs the pre-compiled BBPSSW circuit on the given simulator (or on
        the base simulator, without aging, if none is provided).
        """
        sim = simulator if simulator is not None else self.simulator
        result = sim.run(self._compiled_circuit, shots=self.shots).result()
        counts = result.get_counts()

        success_counts = counts.get("00", 0) + counts.get("11", 0)
        success_rate = success_counts / self.shots
        return success_rate, counts


### 1.7 `qrepeater_twin/orchestrator.py` -- `DigitalTwinOrchestrator`

`run_intelligent` now times the forward pass with `InferenceTimer`
(hardware-accurate on GPU) instead of plain `time.perf_counter()`.
`run_blind_baseline` still never calls the neural network.

In [ ]:
%%writefile qrepeater_twin/orchestrator.py
"""
Component 4 -- Orchestrator (`DigitalTwinOrchestrator`).

Keeps two strictly separate methods, each with its own profiling regime:

    - run_intelligent      : timer isolated around `self.model(x)`,
                               measuring only tau_inf (the forward pass),
                               via `InferenceTimer` (CUDA Events on GPU,
                               `perf_counter` on CPU -- see timing.py).
                               Applies the admission control
                               (HALT_PURIFICATION vs PURIFY).
    - run_blind_baseline    : NEVER invokes the neural network. Classical
                               latency is forced to 0.0 and admission is
                               unconditional.
"""

from __future__ import annotations

import torch
import torch.nn as nn

from .quantum_node import QuantumRepeaterNode
from .timing import InferenceTimer


class DigitalTwinOrchestrator:
    """
    Central orchestrator of the Quantum Repeater Digital Twin.

    Keeps two strictly separate simulation loops to eliminate any
    cross-contamination in profiling between the intelligent (predictive)
    approach and the blind/reactive (baseline) approach.
    """

    def __init__(self, model: nn.Module, quantum_node: QuantumRepeaterNode,
                 threshold: float = 0.65, success_rate_cutoff: float = 0.5,
                 device: torch.device = None):
        self.model = model
        self.quantum_node = quantum_node
        self.threshold = threshold
        self.success_rate_cutoff = success_rate_cutoff
        self.device = device if device is not None else torch.device("cpu")
        self.log = []

    def run_intelligent(self, X_test: torch.Tensor, y_test: torch.Tensor) -> dict:
        """
        Simulation loop with predictive admission control (EdgeLSTM + CS_MSELoss).

        `InferenceTimer` wraps STRICTLY the `self.model(x_sample)` call --
        no other operation (scalar extraction via `.item()`, threshold
        comparison, call into the quantum dataplane) enters the timed
        window. On GPU, the measurement is done with `torch.cuda.Event`
        (hardware markers on the CUDA stream), avoiding the host-clock
        jitter inherent to `time.perf_counter()`; on CPU, `InferenceTimer`
        falls back to `perf_counter()`, since there is no asynchronous
        stream to instrument.
        """
        assert self.model is not None, "run_intelligent requires a trained model."
        self.model.eval()

        results = []
        useful_pairs = 0
        halted = 0
        total_forward_latency = 0.0
        total_steps = len(X_test)

        with torch.no_grad():
            for i in range(total_steps):
                x_sample = X_test[i:i + 1]
                true_fidelity = float(y_test[i].item())

                # --- Isolated profiling: times STRICTLY the forward pass ---
                with InferenceTimer(self.device) as timer:
                    pred_tensor = self.model(x_sample)
                tau_inf = timer.elapsed_s
                # --- End of timed window ---

                pred_fidelity = float(pred_tensor.item())
                total_forward_latency += tau_inf

                if pred_fidelity < self.threshold:
                    halted += 1
                    results.append({
                        "step": i, "action": "HALT_PURIFICATION",
                        "pred_fidelity": pred_fidelity, "true_fidelity": true_fidelity,
                        "latency_s": tau_inf,
                    })
                    continue

                # Approved: dispatches the isolated classical latency to the
                # quantum node (memory aging) and runs the purification circuit.
                aged_simulator = self.quantum_node.apply_latency_decay(tau_inf)
                success_rate, _counts = self.quantum_node.run_purification(simulator=aged_simulator)

                is_useful = (success_rate >= self.success_rate_cutoff) and (true_fidelity >= self.threshold)
                if is_useful:
                    useful_pairs += 1

                results.append({
                    "step": i, "action": "PURIFY",
                    "pred_fidelity": pred_fidelity, "true_fidelity": true_fidelity,
                    "latency_s": tau_inf, "purification_success_rate": success_rate,
                    "useful": is_useful,
                })

        self.log = results
        return {
            "mode": "intelligent",
            "total_steps": total_steps,
            "useful_pairs": useful_pairs,
            "halted": halted,
            "attempted": total_steps - halted,
            "avg_classical_latency_s": total_forward_latency / max(total_steps, 1),
        }

    def run_blind_baseline(self, X_test: torch.Tensor, y_test: torch.Tensor) -> dict:
        """
        Simulation loop for the blind/reactive (baseline) approach.

        Admission is UNCONDITIONAL: every window is purified, with no
        consultation of the predictive model whatsoever. The neural network
        is NEVER instantiated nor called in this routine -- there is no
        residual "background call". Classical latency is therefore
        correctly forced and recorded as 0.0 seconds (there is no inference
        wait to time).
        """
        results = []
        useful_pairs = 0
        total_steps = len(X_test)
        forced_latency = 0.0  # No AI inference => no memory wait.

        for i in range(total_steps):
            true_fidelity = float(y_test[i].item())

            aged_simulator = self.quantum_node.apply_latency_decay(forced_latency)
            success_rate, _counts = self.quantum_node.run_purification(simulator=aged_simulator)

            is_useful = (success_rate >= self.success_rate_cutoff) and (true_fidelity >= self.threshold)
            if is_useful:
                useful_pairs += 1

            results.append({
                "step": i, "action": "PURIFY_BLIND",
                "true_fidelity": true_fidelity, "latency_s": forced_latency,
                "purification_success_rate": success_rate, "useful": is_useful,
            })

        self.log = results
        return {
            "mode": "blind",
            "total_steps": total_steps,
            "useful_pairs": useful_pairs,
            "halted": 0,
            "attempted": total_steps,
            "avg_classical_latency_s": forced_latency,
        }


### 1.8 `qrepeater_twin/pareto_sweep.py` -- `run_pareto_sweep` (multi-seed averaging)

Central fix for the statistical fragility flaw: each `lambda_penalty` is
trained/evaluated over `len(seeds)` independent rounds (5 by default), and
the final table reports mean ± standard deviation per metric instead of a
single point estimate.

In [ ]:
%%writefile qrepeater_twin/pareto_sweep.py
"""
Pareto Frontier -- `lambda_penalty` sweep with multi-seed averaging.

Fixes the "Statistical Fragility" flaw: the original prototype trained the
EdgeLSTM with a single batch (full-batch) and a single random seed per
lambda value, merely *acknowledging* in markdown that individual points
could get stuck in local optima -- without actually mitigating the problem.

Here, each `lambda_penalty` value is trained and evaluated `len(seeds)`
times independently (a different seed each round, applied before the
EdgeLSTM's weight initialization). The point reported on the Pareto
Frontier is the mean across those rounds, accompanied by the standard
deviation -- which makes the training variance at each lambda visible
instead of hidden, drastically reducing the risk of making a production
decision (which lambda to deploy) based on a single, unrepresentative
local optimum.
"""

from __future__ import annotations

import statistics as stats
from typing import List, Sequence

import pandas as pd
import torch

from .models import EdgeLSTM, train_edge_lstm
from .orchestrator import DigitalTwinOrchestrator
from .quantum_node import QuantumRepeaterNode


def _mean_std(values: Sequence[float]) -> tuple:
    mean = stats.fmean(values)
    std = stats.pstdev(values) if len(values) > 1 else 0.0
    return mean, std


def run_pareto_sweep(lambda_values: list, X_train: torch.Tensor, y_train: torch.Tensor,
                      X_test: torch.Tensor, y_test: torch.Tensor, device: torch.device,
                      threshold: float = 0.65, epochs: int = 120, lr: float = 3e-3,
                      hidden_size: int = 16, T1: float = 50e-6, T2: float = 30e-6,
                      depol_prob: float = 0.01, shots: int = 512, quantum_seed: int = 7,
                      lambda_fn: float = 2.0, discard_penalty_weight: float = 5.0,
                      max_discard_rate: float = 0.60, seeds: List[int] = None):
    """
    Runs the Pareto Frontier sweep over the `lambda_penalty` hyperparameter
    of CS_MSELoss, with multi-seed averaging at every point.

    Parameters
    ----------
    seeds : list[int], optional
        Seeds used to repeat training/evaluation at each lambda value.
        Default: 5 seeds ([42, 43, 44, 45, 46]). Each round trains an
        EdgeLSTM from scratch (weight initialization determined by the
        seed) and runs the full Digital Twin over the test set; the
        results of the `len(seeds)` rounds are aggregated into mean ±
        standard deviation before composing that lambda's row in the
        final table.

    Returns
    -------
    results_df : pd.DataFrame
        Consolidated table with one row per lambda value, reporting the
        mean ± standard deviation of each metric across seeds.
    baseline_metrics : dict
        Blind/reactive baseline metrics, computed exactly once (they don't
        depend on lambda or seed, since the neural network is never
        invoked).
    per_seed_results : dict[float, list[dict]]
        Raw metrics from each individual round (lambda -> list of dicts),
        preserved for auditing/debugging and to allow recomputing other
        statistics (median, confidence intervals, etc.) without retraining.
    """
    if seeds is None:
        seeds = [42, 43, 44, 45, 46]

    # --- Blind/reactive baseline: computed exactly once, independent of lambda and seed ---
    print("Running blind/reactive baseline (unconditional admission, forced latency = 0.0)...")
    baseline_node = QuantumRepeaterNode(T1=T1, T2=T2, depol_prob=depol_prob, shots=shots, seed=quantum_seed)
    baseline_orchestrator = DigitalTwinOrchestrator(model=None, quantum_node=baseline_node,
                                                      threshold=threshold, device=device)
    baseline_metrics = baseline_orchestrator.run_blind_baseline(X_test, y_test)
    print(f"  Baseline: Attempts={baseline_metrics['attempted']} | "
          f"Useful pairs={baseline_metrics['useful_pairs']} | "
          f"Forced latency={baseline_metrics['avg_classical_latency_s']*1000:.4f} ms\n")

    rows = []
    per_seed_results = {}

    for lam in lambda_values:
        print(f"[lambda_penalty={lam}] training EdgeLSTM on {len(seeds)} seeds "
              f"({epochs} epochs each) ...")

        seed_runs = []
        for seed in seeds:
            # Seed applied BEFORE model construction: guarantees each round
            # starts from an independent weight initialization.
            torch.manual_seed(seed)
            model = EdgeLSTM(input_size=2, hidden_size=hidden_size, num_layers=1).to(device)
            model = train_edge_lstm(
                model, X_train, y_train,
                threshold=threshold, lambda_penalty=lam, lambda_fn=lambda_fn,
                discard_penalty_weight=discard_penalty_weight, max_discard_rate=max_discard_rate,
                epochs=epochs, lr=lr, device=device, seed=seed, verbose=False,
            )

            # Each round uses its own QuantumRepeaterNode with the SAME
            # quantum_seed: isolates the observed variation to the
            # EdgeLSTM's initialization/training, not to the quantum
            # simulator (which must remain comparable across seeds and
            # across lambdas).
            quantum_node = QuantumRepeaterNode(T1=T1, T2=T2, depol_prob=depol_prob,
                                                shots=shots, seed=quantum_seed)
            orchestrator = DigitalTwinOrchestrator(model=model, quantum_node=quantum_node,
                                                     threshold=threshold, device=device)
            metrics = orchestrator.run_intelligent(X_test, y_test)

            yield_qpu_pct = (metrics["useful_pairs"] / max(metrics["attempted"], 1)) * 100.0
            deficit_surplus = metrics["useful_pairs"] - baseline_metrics["useful_pairs"]

            seed_runs.append({
                "seed": seed,
                "halted": metrics["halted"],
                "attempted": metrics["attempted"],
                "useful_pairs": metrics["useful_pairs"],
                "yield_qpu_pct": yield_qpu_pct,
                "deficit_surplus": deficit_surplus,
                "avg_inference_latency_ms": metrics["avg_classical_latency_s"] * 1000.0,
            })

        per_seed_results[lam] = seed_runs

        halted_mean, halted_std = _mean_std([r["halted"] for r in seed_runs])
        attempted_mean, attempted_std = _mean_std([r["attempted"] for r in seed_runs])
        useful_mean, useful_std = _mean_std([r["useful_pairs"] for r in seed_runs])
        yield_mean, yield_std = _mean_std([r["yield_qpu_pct"] for r in seed_runs])
        deficit_mean, deficit_std = _mean_std([r["deficit_surplus"] for r in seed_runs])
        latency_mean, latency_std = _mean_std([r["avg_inference_latency_ms"] for r in seed_runs])

        rows.append({
            "Lambda": lam,
            "N Seeds": len(seeds),
            "Cycles Saved (HALT)": f"{halted_mean:.1f} +/- {halted_std:.1f}",
            "QPU Attempts": f"{attempted_mean:.1f} +/- {attempted_std:.1f}",
            "Useful Pairs": f"{useful_mean:.1f} +/- {useful_std:.1f}",
            "QPU Yield (%)": f"{yield_mean:.2f} +/- {yield_std:.2f}",
            "SKR Deficit/Surplus": f"{deficit_mean:+.1f} +/- {deficit_std:.1f}",
            "Inference Latency (ms)": f"{latency_mean:.4f} +/- {latency_std:.4f}",
        })

        print(f"  -> QPU Yield (mean +/- std) = {yield_mean:.2f}% +/- {yield_std:.2f}% | "
              f"Deficit/Surplus (mean) = {deficit_mean:+.1f} | "
              f"Inference latency (mean) = {latency_mean:.4f} ms\n")

    results_df = pd.DataFrame(rows, columns=[
        "Lambda", "N Seeds", "Cycles Saved (HALT)", "QPU Attempts",
        "Useful Pairs", "QPU Yield (%)", "SKR Deficit/Surplus",
        "Inference Latency (ms)",
    ])
    return results_df, baseline_metrics, per_seed_results


### 1.9 `qrepeater_twin/baselines.py` -- LSTM+MSE, Random Forest, XGBoost, Transformer

Alternative fidelity predictors, all runnable through the exact same
`DigitalTwinOrchestrator` loop as the `EdgeLSTM + CS_MSELoss` admission
controller: **LSTM+MSE** (same architecture, plain `nn.MSELoss`, isolates
the contribution of the cost-sensitive loss), **Random Forest** and
**XGBoost** (classical, non-recurrent baselines over the flattened window
-- XGBoost is an optional dependency, skipped with a warning if not
installed), and a small **Transformer** encoder (higher-capacity
architectural baseline).

In [ ]:
%%writefile qrepeater_twin/baselines.py
"""
Component 5 -- Predictor baselines compared against the intelligent
admission controller (EdgeLSTM + CS_MSELoss).

Every baseline here ends up wrapped so it exposes the exact same runtime
contract the rest of the pipeline already relies on
(`orchestrator.DigitalTwinOrchestrator.run_intelligent`):

    - `.eval()`                       (no-op for non-torch models)
    - `model(x)` with `x` of shape (1, window_size, n_features)
      returning something with a scalar `.item()` in [0, 1]

This lets `DigitalTwinOrchestrator` -- and therefore `InferenceTimer`,
`QuantumRepeaterNode.apply_latency_decay`, and every downstream metric --
run *unmodified* regardless of whether the underlying predictor is a
recurrent network, a tree ensemble, or a Transformer. Comparability
across architectures was the whole point of adding these baselines: they
all get admitted into the digital twin loop through the same door.

Baselines implemented:

    1. LSTM + MSE        : `train_lstm_mse` -- the *same* `EdgeLSTM`
                             architecture as the intelligent controller,
                             trained with a plain, cost-insensitive
                             `nn.MSELoss`. Isolates the contribution of
                             `CS_MSELoss` itself (architecture held fixed).
    2. Random Forest      : `RandomForestFidelityModel` -- classical,
                             non-recurrent ensemble over the flattened
                             window.
    3. XGBoost            : `XGBoostFidelityModel` -- gradient-boosted
                             trees over the flattened window. Optional
                             dependency: raises a clear, catchable
                             `ImportError` (not a hard crash) if `xgboost`
                             isn't installed, so callers can skip it.
    4. Transformer        : `TinyTransformer` + `train_transformer` -- a
                             small Transformer encoder over the same input
                             window, trained with plain `nn.MSELoss`, as a
                             higher-capacity architectural baseline.
"""

from __future__ import annotations

import math

import numpy as np
import torch
import torch.nn as nn


# ---------------------------------------------------------------------------
# 1) LSTM + plain MSE (same architecture as EdgeLSTM, cost-insensitive loss)
# ---------------------------------------------------------------------------

def train_lstm_mse(model: nn.Module, X_train: torch.Tensor, y_train: torch.Tensor,
                    epochs: int = 150, lr: float = 0.012, device: torch.device = None,
                    seed: int = None, verbose: bool = False) -> nn.Module:
    """
    Trains an `EdgeLSTM` (or any compatible module) with a plain
    `nn.MSELoss`, i.e. WITHOUT the False-Positive-severe, cost-sensitive
    weighting of `CS_MSELoss`.

    Kept architecturally identical to `models.train_edge_lstm` (same
    full-batch loop, same seed-before-init convention) so that any
    difference observed downstream (QPU yield, useful pairs, latency) is
    attributable to the *loss function*, not to incidental differences in
    the training procedure.
    """
    if seed is not None:
        torch.manual_seed(seed)

    if device is not None:
        model = model.to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()
        if verbose and (epoch + 1) % 30 == 0:
            print(f"    [LSTM+MSE] Epoch {epoch + 1:3d}/{epochs} | MSE Loss: {loss.item():.6f}")
    return model


# ---------------------------------------------------------------------------
# Shared helpers: window flattening for the non-recurrent baselines
# ---------------------------------------------------------------------------

def _flatten_windows(X: torch.Tensor) -> np.ndarray:
    """(batch, seq_len, n_features) -> (batch, seq_len * n_features), on CPU/numpy."""
    X_np = X.detach().cpu().numpy() if isinstance(X, torch.Tensor) else np.asarray(X)
    return X_np.reshape(X_np.shape[0], -1)


class _SklearnRegressorAdapter:
    """
    Wraps a fitted scikit-learn-style regressor (`.predict(X)`) so it can
    be dropped into `DigitalTwinOrchestrator.run_intelligent` exactly like
    a `torch.nn.Module`: same `.eval()` / callable contract, same
    (1, 1)-shaped, [0, 1]-clipped tensor output.

    These models have no notion of `torch` autograd or CUDA, so
    `InferenceTimer` in `run_intelligent` transparently falls back to
    `perf_counter()` for them (its CUDA-Events branch is only entered when
    `device.type == "cuda"`, and this adapter always returns a CPU
    tensor) -- still a fair, isolated latency measurement of exactly the
    `.predict(...)` call, nothing else.
    """

    def __init__(self, fitted_estimator, name: str):
        self._estimator = fitted_estimator
        self.name = name

    def eval(self) -> "_SklearnRegressorAdapter":
        return self  # stateless at inference time; kept for interface parity

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        x_flat = _flatten_windows(x)
        pred = self._estimator.predict(x_flat)
        pred = np.clip(pred, 0.0, 1.0)
        return torch.as_tensor(pred, dtype=torch.float32).reshape(-1, 1)


# ---------------------------------------------------------------------------
# 2) Random Forest
# ---------------------------------------------------------------------------

class RandomForestFidelityModel:
    """
    Random Forest regressor baseline for F_hat(t), trained on the
    flattened (window_size * n_features) feature vector.

    Unlike `EdgeLSTM`, this model has no notion of sequence order beyond
    whatever the tree splits pick up from the flattened positions -- a
    useful contrast against the recurrent baselines: does explicit
    temporal structure (LSTM) actually earn its keep over an
    order-agnostic ensemble with the same raw information?
    """

    def __init__(self, n_estimators: int = 200, max_depth: int = 8, seed: int = 42):
        from sklearn.ensemble import RandomForestRegressor

        self.model = RandomForestRegressor(
            n_estimators=n_estimators, max_depth=max_depth,
            random_state=seed, n_jobs=-1,
        )

    def fit(self, X_train: torch.Tensor, y_train: torch.Tensor) -> "RandomForestFidelityModel":
        X_flat = _flatten_windows(X_train)
        y_flat = _flatten_windows(y_train).ravel()
        self.model.fit(X_flat, y_flat)
        return self

    def as_orchestrator_model(self) -> _SklearnRegressorAdapter:
        """Wraps the fitted estimator for use with `DigitalTwinOrchestrator`."""
        return _SklearnRegressorAdapter(self.model, name="RandomForest")


def train_random_forest(X_train: torch.Tensor, y_train: torch.Tensor,
                         n_estimators: int = 200, max_depth: int = 8,
                         seed: int = 42) -> _SklearnRegressorAdapter:
    """Convenience one-shot: fit a Random Forest and return it orchestrator-ready."""
    rf = RandomForestFidelityModel(n_estimators=n_estimators, max_depth=max_depth, seed=seed)
    rf.fit(X_train, y_train)
    return rf.as_orchestrator_model()


# ---------------------------------------------------------------------------
# 3) XGBoost (optional dependency)
# ---------------------------------------------------------------------------

class XGBoostFidelityModel:
    """
    Gradient-boosted trees (XGBoost) baseline for F_hat(t), trained on the
    same flattened window representation as `RandomForestFidelityModel`.

    `xgboost` is an OPTIONAL dependency (not in `requirements.txt` by
    default): the import is attempted lazily, inside `__init__`, and
    raises a plain `ImportError` with an actionable message
    (`pip install xgboost`) rather than crashing the whole comparison run.
    Callers (see `model_comparison.run_model_comparison`) catch this and
    skip the XGBoost row, logging a warning instead of failing the sweep.
    """

    def __init__(self, n_estimators: int = 200, max_depth: int = 5,
                 learning_rate: float = 0.1, seed: int = 42):
        try:
            from xgboost import XGBRegressor
        except ImportError as exc:  # pragma: no cover - exercised only when xgboost is absent
            raise ImportError(
                "XGBoost baseline requested but the 'xgboost' package is not installed. "
                "Install it with `pip install xgboost` or set "
                "ComparisonConfig.include_xgboost=False to skip this baseline."
            ) from exc

        self.model = XGBRegressor(
            n_estimators=n_estimators, max_depth=max_depth,
            learning_rate=learning_rate, objective="reg:squarederror",
            random_state=seed, n_jobs=-1,
        )

    def fit(self, X_train: torch.Tensor, y_train: torch.Tensor) -> "XGBoostFidelityModel":
        X_flat = _flatten_windows(X_train)
        y_flat = _flatten_windows(y_train).ravel()
        self.model.fit(X_flat, y_flat)
        return self

    def as_orchestrator_model(self) -> _SklearnRegressorAdapter:
        return _SklearnRegressorAdapter(self.model, name="XGBoost")


def train_xgboost(X_train: torch.Tensor, y_train: torch.Tensor,
                   n_estimators: int = 200, max_depth: int = 5,
                   learning_rate: float = 0.1, seed: int = 42) -> _SklearnRegressorAdapter:
    """Convenience one-shot: fit an XGBoost regressor and return it orchestrator-ready.

    Raises `ImportError` if `xgboost` is not installed -- see
    `XGBoostFidelityModel`.
    """
    xgb = XGBoostFidelityModel(n_estimators=n_estimators, max_depth=max_depth,
                                learning_rate=learning_rate, seed=seed)
    xgb.fit(X_train, y_train)
    return xgb.as_orchestrator_model()


# ---------------------------------------------------------------------------
# 4) Transformer encoder
# ---------------------------------------------------------------------------

class _PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding (Vaswani et al., 2017)."""

    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32)
                              * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term[: pe[:, 1::2].shape[1]])
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1), :]


class TinyTransformer(nn.Module):
    """
    Small Transformer-encoder baseline for F_hat(t), operating on the same
    (batch, seq_len, n_features) windows as `EdgeLSTM`.

    Architecture:
        input projection (Linear: n_features -> d_model)
        -> sinusoidal positional encoding
        -> `num_layers` x TransformerEncoderLayer (self-attention + FFN)
        -> mean-pool over the sequence dimension
        -> linear head + sigmoid, producing F_hat(t) in [0, 1]

    A higher-capacity, attention-based architectural baseline against
    which the compact, recurrent `EdgeLSTM` (designed for cheap edge
    inference) can be compared on the yield/latency/energy trade-off,
    not just on raw predictive accuracy.
    """

    def __init__(self, input_size: int = 2, d_model: int = 32, nhead: int = 4,
                 num_layers: int = 2, dim_feedforward: int = 64, dropout: float = 0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        self.pos_encoding = _PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, 1)
        self.activation = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.input_proj(x)
        h = self.pos_encoding(h)
        h = self.encoder(h)
        pooled = h.mean(dim=1)
        pred = self.activation(self.head(pooled))
        return pred


def train_transformer(model: nn.Module, X_train: torch.Tensor, y_train: torch.Tensor,
                       epochs: int = 150, lr: float = 0.005, device: torch.device = None,
                       seed: int = None, verbose: bool = False) -> nn.Module:
    """
    Trains a `TinyTransformer` with a plain `nn.MSELoss`, following the
    same full-batch, seed-before-init convention as `train_edge_lstm` /
    `train_lstm_mse` so results stay directly comparable.
    """
    if seed is not None:
        torch.manual_seed(seed)

    if device is not None:
        model = model.to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()
        if verbose and (epoch + 1) % 30 == 0:
            print(f"    [Transformer] Epoch {epoch + 1:3d}/{epochs} | MSE Loss: {loss.item():.6f}")
    return model


### 1.10 `qrepeater_twin/metrics.py` -- throughput, QPU economy, energy, decision matrix

Derived operational metrics needed to actually choose a predictor for
deployment: useful-pair **throughput** (pairs/s), **QPU economy**
(purification attempts/shots avoided vs. the blind baseline), an
illustrative **energy** accounting (quantum gate/shot energy + classical
inference power, via `EnergyConfig`), and `build_decision_matrix`, which
normalizes every weighted criterion (direction-aware) and ranks candidate
models by a composite score.

In [ ]:
%%writefile qrepeater_twin/metrics.py
"""
Component 6 -- Derived operational metrics and the multi-criteria
decision matrix.

`pareto_sweep.py` already reports QPU yield, useful-pair deficit/surplus,
and inference latency. This module adds the three metrics needed to
actually *decide* which predictor to deploy, plus the decision matrix
itself:

    - `compute_throughput`     : useful entangled pairs delivered per
                                   second of wall-clock operation.
    - `compute_qpu_economy`    : purification attempts (and therefore QPU
                                   time/shots) avoided relative to the
                                   blind/reactive baseline.
    - `compute_energy_report`  : an illustrative Joules accounting of the
                                   quantum-side (per purification attempt)
                                   and classical-side (per prediction)
                                   energy cost, and the resulting savings
                                   relative to the blind baseline.
    - `build_decision_matrix`  : normalizes every criterion (direction-
                                   aware) across candidate models and
                                   computes a weighted composite score.

All three "compute_*" functions accept the same `metrics` dict produced
by `DigitalTwinOrchestrator.run_intelligent` / `run_blind_baseline`
(`total_steps`, `useful_pairs`, `halted`, `attempted`,
`avg_classical_latency_s`), so they compose directly with the existing
pipeline without touching `orchestrator.py`.
"""

from __future__ import annotations

from typing import Dict, List, Sequence

import pandas as pd

from .config import EnergyConfig


# ---------------------------------------------------------------------------
# Throughput
# ---------------------------------------------------------------------------

def compute_throughput(metrics: dict, cycle_time_s: float = 1e-3) -> dict:
    """
    Useful-pair throughput, in pairs/second.

    `cycle_time_s` is the fixed repetition period of one WDM
    entanglement-generation attempt (the channel keeps ticking at this
    rate regardless of the admission decision -- HALT just skips the
    purification step for that cycle, it doesn't stop the clock). Total
    wall-clock time is therefore:

        total_time_s = total_steps * cycle_time_s + total_classical_latency_s

    where `total_classical_latency_s = avg_classical_latency_s *
    total_steps` adds back the classical inference overhead paid on every
    cycle (zero for the blind baseline, which never invokes a predictor).

    Returns a dict with `total_time_s` and `throughput_pairs_per_s`.
    """
    total_steps = max(metrics["total_steps"], 1)
    total_classical_latency_s = metrics["avg_classical_latency_s"] * total_steps
    total_time_s = total_steps * cycle_time_s + total_classical_latency_s
    throughput = metrics["useful_pairs"] / total_time_s if total_time_s > 0 else 0.0
    return {
        "total_time_s": total_time_s,
        "throughput_pairs_per_s": throughput,
    }


# ---------------------------------------------------------------------------
# QPU economy
# ---------------------------------------------------------------------------

def compute_qpu_economy(metrics: dict, baseline_metrics: dict, shots_per_attempt: int = None) -> dict:
    """
    QPU-time economy relative to the blind/reactive baseline.

    The blind baseline purifies unconditionally on every cycle
    (`baseline_metrics["attempted"] == baseline_metrics["total_steps"]`).
    A predictive controller instead HALTs low-fidelity cycles, so its
    `attempted` count -- and therefore the number of BBPSSW circuits
    actually dispatched to the QPU -- is lower. This function reports both
    the absolute and percentage reduction in QPU attempts (a direct proxy
    for QPU-time budget saved), alongside the resulting useful-pair
    deficit/surplus already computed in `pareto_sweep.py` for context.

    `shots_per_attempt` (typically `QuantumConfig.shots`) is optional: when
    given, the avoided attempts are also translated into avoided QPU shots
    (`qpu_shots_saved`); when omitted, `qpu_shots_saved` is left as `None`.
    """
    baseline_attempted = max(baseline_metrics["attempted"], 1)
    cycles_saved = baseline_metrics["attempted"] - metrics["attempted"]
    cycles_saved_pct = (cycles_saved / baseline_attempted) * 100.0
    shots_saved = cycles_saved * shots_per_attempt if shots_per_attempt is not None else None
    return {
        "qpu_cycles_saved": cycles_saved,
        "qpu_cycles_saved_pct": cycles_saved_pct,
        "qpu_shots_saved": shots_saved,
        "useful_pairs_deficit_surplus": metrics["useful_pairs"] - baseline_metrics["useful_pairs"],
    }


# ---------------------------------------------------------------------------
# Energy accounting
# ---------------------------------------------------------------------------

def _energy_per_attempt_j(energy_cfg: EnergyConfig, shots: int) -> float:
    """
    Illustrative quantum-side energy of ONE BBPSSW purification attempt
    (see `EnergyConfig` docstring for the caveat that these are
    order-of-magnitude coefficients, not datasheet values):

        E_attempt = shots * (n_1q * E_1q + n_2q * E_2q + E_shot_overhead)
    """
    per_shot = (
        energy_cfg.gates_1q_per_attempt * energy_cfg.joules_per_1q_gate
        + energy_cfg.gates_2q_per_attempt * energy_cfg.joules_per_2q_gate
        + energy_cfg.joules_per_shot_overhead
    )
    return shots * per_shot


def compute_energy_report(metrics: dict, baseline_metrics: dict, shots: int,
                           energy_cfg: EnergyConfig = None) -> dict:
    """
    Illustrative total-energy comparison (Joules) between a predictive
    controller and the blind baseline, over the same evaluation window.

    Two components, summed per cycle:

        - Quantum energy : paid ONLY on cycles where a purification
          attempt is actually dispatched (`attempted`), at
          `_energy_per_attempt_j(...)` each.
        - Classical energy : paid on EVERY cycle. For a predictive
          controller, `classical_inference_power_w * avg_classical_latency_s`
          per cycle (the edge accelerator actively running the predictor);
          for the blind baseline (no predictor invoked),
          `classical_idle_power_w * cycle-equivalent latency` -- using the
          predictor's own average latency as the reference cycle length so
          the two totals are computed over comparable wall-clock time.

    Returns quantum/classical/total energy (J) for `metrics`, the
    equivalent total for the baseline, and the resulting percentage
    saved (positive = the predictor used less total energy than blind
    purification).
    """
    energy_cfg = energy_cfg or EnergyConfig()
    e_attempt = _energy_per_attempt_j(energy_cfg, shots)

    quantum_j = metrics["attempted"] * e_attempt
    classical_j = (
        metrics["total_steps"] * metrics["avg_classical_latency_s"]
        * energy_cfg.classical_inference_power_w
    )
    total_j = quantum_j + classical_j

    baseline_quantum_j = baseline_metrics["attempted"] * e_attempt
    # Reference idle window: the predictor's own average latency, so the
    # baseline's classical term is on the same time basis as `metrics`
    # rather than implicitly zero.
    baseline_classical_j = (
        baseline_metrics["total_steps"] * metrics["avg_classical_latency_s"]
        * energy_cfg.classical_idle_power_w
    )
    baseline_total_j = baseline_quantum_j + baseline_classical_j

    energy_saved_j = baseline_total_j - total_j
    energy_saved_pct = (energy_saved_j / baseline_total_j * 100.0) if baseline_total_j > 0 else 0.0

    return {
        "quantum_energy_j": quantum_j,
        "classical_energy_j": classical_j,
        "total_energy_j": total_j,
        "baseline_total_energy_j": baseline_total_j,
        "energy_saved_j": energy_saved_j,
        "energy_saved_pct": energy_saved_pct,
    }


# ---------------------------------------------------------------------------
# Decision matrix
# ---------------------------------------------------------------------------

# Criteria where a LOWER raw value is better (inverted before weighting).
_COST_CRITERIA = {"inference_latency_ms", "total_energy_j"}


def build_decision_matrix(rows: List[dict], weights: Dict[str, float],
                           model_key: str = "Model") -> pd.DataFrame:
    """
    Multi-criteria decision matrix: normalizes every weighted criterion
    to [0, 1] (min-max, direction-aware -- see `_COST_CRITERIA`) across
    the candidate models in `rows`, then computes a weighted composite
    score in [0, 1] (higher is better).

    Parameters
    ----------
    rows : list[dict]
        One dict per candidate model, each containing `model_key` plus
        every key present in `weights` (raw, un-normalized values -- e.g.
        `qpu_yield_pct`, `throughput_pairs_per_s`, `qpu_cycles_saved_pct`,
        `energy_saved_pct`, `inference_latency_ms`).
    weights : dict[str, float]
        Criterion -> weight (non-negative; renormalized to sum to 1
        internally, so callers don't need to pre-normalize).
    model_key : str
        Column name identifying each row's model (default "Model").

    Returns
    -------
    pd.DataFrame
        One row per model, with a `<criterion>_norm` column per weighted
        criterion, a `Decision Score` column (weighted sum, higher =
        better), and a `Rank` column (1 = best), sorted by rank.
    """
    if not rows:
        return pd.DataFrame(columns=[model_key, "Decision Score", "Rank"])

    total_weight = sum(max(w, 0.0) for w in weights.values())
    if total_weight <= 0:
        raise ValueError("build_decision_matrix: weights must sum to a positive value.")
    norm_weights = {k: max(w, 0.0) / total_weight for k, w in weights.items()}

    df = pd.DataFrame(rows)
    scored = df.copy()

    for criterion in norm_weights:
        if criterion not in df.columns:
            raise KeyError(f"build_decision_matrix: missing criterion '{criterion}' in rows.")
        col = df[criterion].astype(float)
        lo, hi = col.min(), col.max()
        span = hi - lo
        if span == 0:
            normalized = pd.Series([1.0] * len(col), index=col.index)
        else:
            normalized = (col - lo) / span
            if criterion in _COST_CRITERIA:
                normalized = 1.0 - normalized
        scored[f"{criterion}_norm"] = normalized

    scored["Decision Score"] = sum(
        scored[f"{criterion}_norm"] * w for criterion, w in norm_weights.items()
    )
    scored["Rank"] = scored["Decision Score"].rank(ascending=False, method="min").astype(int)

    ordered_cols = [model_key] + [f"{c}_norm" for c in norm_weights] + ["Decision Score", "Rank"]
    remaining_cols = [c for c in scored.columns if c not in ordered_cols]
    scored = scored[ordered_cols + remaining_cols]
    return scored.sort_values("Rank").reset_index(drop=True)


### 1.11 `qrepeater_twin/model_comparison.py` -- `run_model_comparison`

Trains/evaluates every predictor baseline above -- plus
`EdgeLSTM+CS-MSE` at one representative `lambda_penalty` and the blind
baseline -- over the same multi-seed protocol as `run_pareto_sweep`, and
returns the consolidated results table and the ranked decision matrix.

In [ ]:
%%writefile qrepeater_twin/model_comparison.py
"""
Component 7 -- Cross-architecture model comparison
(`run_model_comparison`).

Extends the single-hyperparameter Pareto sweep (`pareto_sweep.py`, which
only varies `lambda_penalty` for a fixed `EdgeLSTM + CS_MSELoss`) to a
comparison ACROSS predictor architectures/objectives:

    - EdgeLSTM + CS_MSELoss  (at one representative `lambda_penalty`,
                                `ComparisonConfig.representative_lambda`)
    - LSTM + MSE              (`baselines.train_lstm_mse`)
    - Random Forest            (`baselines.train_random_forest`)
    - XGBoost                   (`baselines.train_xgboost`, skipped with a
                                  warning if not installed)
    - Transformer                (`baselines.train_transformer`)
    - Blind/reactive baseline     (`orchestrator.run_blind_baseline`,
                                    computed once, same as in
                                    `pareto_sweep.py`)

Every predictor is trained/evaluated over the SAME `seeds` and run
through the SAME `DigitalTwinOrchestrator.run_intelligent` loop against
the SAME `QuantumRepeaterNode` configuration -- so the resulting
throughput / QPU-economy / energy / latency numbers are directly
comparable, and `metrics.build_decision_matrix` can rank them on equal
footing.
"""

from __future__ import annotations

import statistics as stats
import warnings
from typing import List, Sequence

import pandas as pd
import torch

from .baselines import TinyTransformer, train_lstm_mse, train_random_forest, train_transformer
from .config import BaselineConfig, ComparisonConfig, EnergyConfig, QuantumConfig, TrainConfig
from .models import EdgeLSTM, train_edge_lstm
from .metrics import build_decision_matrix, compute_energy_report, compute_qpu_economy, compute_throughput
from .orchestrator import DigitalTwinOrchestrator
from .quantum_node import QuantumRepeaterNode


def _mean_std(values: Sequence[float]) -> tuple:
    mean = stats.fmean(values)
    std = stats.pstdev(values) if len(values) > 1 else 0.0
    return mean, std


def _build_predictor(name: str, seed: int, device: torch.device,
                      X_train: torch.Tensor, y_train: torch.Tensor,
                      train_cfg: TrainConfig, baseline_cfg: BaselineConfig,
                      representative_lambda: float):
    """
    Trains one predictor of `name` for one `seed` and returns an object
    compatible with `DigitalTwinOrchestrator` (a trained `nn.Module`, or a
    `_SklearnRegressorAdapter` for the tree-ensemble baselines).

    Raises `ImportError` for "XGBoost" when `xgboost` isn't installed --
    callers are expected to catch this and skip the model (see
    `run_model_comparison`).
    """
    torch.manual_seed(seed)

    if name == "EdgeLSTM+CS-MSE":
        model = EdgeLSTM(input_size=2, hidden_size=train_cfg.hidden_size, num_layers=1).to(device)
        return train_edge_lstm(
            model, X_train, y_train, threshold=train_cfg.threshold,
            lambda_penalty=representative_lambda, lambda_fn=train_cfg.lambda_fn,
            discard_penalty_weight=train_cfg.discard_penalty_weight,
            max_discard_rate=train_cfg.max_discard_rate,
            epochs=train_cfg.epochs, lr=train_cfg.lr, device=device, seed=seed,
        )

    if name == "LSTM+MSE":
        model = EdgeLSTM(input_size=2, hidden_size=baseline_cfg.lstm_mse_hidden_size, num_layers=1).to(device)
        return train_lstm_mse(model, X_train, y_train, epochs=baseline_cfg.lstm_mse_epochs,
                               lr=baseline_cfg.lstm_mse_lr, device=device, seed=seed)

    if name == "RandomForest":
        return train_random_forest(X_train, y_train, n_estimators=baseline_cfg.rf_n_estimators,
                                    max_depth=baseline_cfg.rf_max_depth, seed=seed)

    if name == "XGBoost":
        from .baselines import train_xgboost  # local import: surfaces ImportError to the caller
        return train_xgboost(X_train, y_train, n_estimators=baseline_cfg.xgb_n_estimators,
                              max_depth=baseline_cfg.xgb_max_depth,
                              learning_rate=baseline_cfg.xgb_learning_rate, seed=seed)

    if name == "Transformer":
        model = TinyTransformer(
            input_size=2, d_model=baseline_cfg.transformer_d_model,
            nhead=baseline_cfg.transformer_nhead, num_layers=baseline_cfg.transformer_num_layers,
            dim_feedforward=baseline_cfg.transformer_dim_feedforward,
        ).to(device)
        return train_transformer(model, X_train, y_train, epochs=baseline_cfg.transformer_epochs,
                                  lr=baseline_cfg.transformer_lr, device=device, seed=seed)

    raise ValueError(f"Unknown predictor name: {name!r}")


def run_model_comparison(X_train: torch.Tensor, y_train: torch.Tensor,
                          X_test: torch.Tensor, y_test: torch.Tensor, device: torch.device,
                          train_cfg: TrainConfig = None, quantum_cfg: QuantumConfig = None,
                          baseline_cfg: BaselineConfig = None, energy_cfg: EnergyConfig = None,
                          comparison_cfg: ComparisonConfig = None,
                          model_names: List[str] = None):
    """
    Trains/evaluates every predictor baseline (plus the blind baseline)
    over `comparison_cfg.seeds`, and reports throughput, QPU economy,
    energy, and a ranked multi-criteria decision matrix.

    Parameters
    ----------
    model_names : list[str], optional
        Subset/order of predictors to include. Default: all five
        (`["EdgeLSTM+CS-MSE", "LSTM+MSE", "RandomForest", "XGBoost",
        "Transformer"]`). "XGBoost" is silently skipped (with a printed
        warning) if `xgboost` isn't installed or
        `comparison_cfg.include_xgboost` is `False`.

    Returns
    -------
    results_df : pd.DataFrame
        One row per model (mean +/- std across seeds): QPU yield, useful
        pairs, throughput, QPU cycles saved, energy saved, inference
        latency.
    baseline_metrics : dict
        Blind/reactive baseline metrics (see `orchestrator.run_blind_baseline`).
    decision_matrix_df : pd.DataFrame
        Output of `metrics.build_decision_matrix` -- normalized criteria,
        weighted `Decision Score`, and `Rank` (1 = recommended model),
        built from the SAME mean values reported in `results_df`.
    per_model_seed_results : dict[str, list[dict]]
        Raw per-seed metrics for each model, preserved for auditing.
    """
    train_cfg = train_cfg or TrainConfig()
    quantum_cfg = quantum_cfg or QuantumConfig()
    baseline_cfg = baseline_cfg or BaselineConfig()
    energy_cfg = energy_cfg or EnergyConfig()
    comparison_cfg = comparison_cfg or ComparisonConfig()
    model_names = model_names or ["EdgeLSTM+CS-MSE", "LSTM+MSE", "RandomForest", "XGBoost", "Transformer"]

    # --- Blind/reactive baseline: computed exactly once ---
    print("Running blind/reactive baseline (unconditional admission, forced latency = 0.0)...")
    baseline_node = QuantumRepeaterNode(T1=quantum_cfg.T1, T2=quantum_cfg.T2, depol_prob=quantum_cfg.depol_prob,
                                         shots=quantum_cfg.shots, seed=quantum_cfg.seed)
    baseline_orchestrator = DigitalTwinOrchestrator(model=None, quantum_node=baseline_node,
                                                      threshold=train_cfg.threshold, device=device)
    baseline_metrics = baseline_orchestrator.run_blind_baseline(X_test, y_test)
    print(f"  Baseline: Attempts={baseline_metrics['attempted']} | "
          f"Useful pairs={baseline_metrics['useful_pairs']}\n")

    rows = []
    per_model_seed_results = {}

    for name in model_names:
        if name == "XGBoost" and not comparison_cfg.include_xgboost:
            print("[XGBoost] skipped (ComparisonConfig.include_xgboost=False).\n")
            continue

        print(f"[{name}] training on {len(comparison_cfg.seeds)} seeds ...")
        seed_runs = []
        skipped = False

        for seed in comparison_cfg.seeds:
            try:
                model = _build_predictor(name, seed, device, X_train, y_train,
                                          train_cfg, baseline_cfg, comparison_cfg.representative_lambda)
            except ImportError as exc:
                warnings.warn(f"[{name}] skipped: {exc}")
                skipped = True
                break

            quantum_node = QuantumRepeaterNode(T1=quantum_cfg.T1, T2=quantum_cfg.T2,
                                                depol_prob=quantum_cfg.depol_prob,
                                                shots=quantum_cfg.shots, seed=quantum_cfg.seed)
            orchestrator = DigitalTwinOrchestrator(model=model, quantum_node=quantum_node,
                                                     threshold=train_cfg.threshold, device=device)
            metrics = orchestrator.run_intelligent(X_test, y_test)

            throughput = compute_throughput(metrics, cycle_time_s=comparison_cfg.cycle_time_s)
            qpu_economy = compute_qpu_economy(metrics, baseline_metrics, shots_per_attempt=quantum_cfg.shots)
            energy = compute_energy_report(metrics, baseline_metrics, shots=quantum_cfg.shots, energy_cfg=energy_cfg)

            yield_qpu_pct = (metrics["useful_pairs"] / max(metrics["attempted"], 1)) * 100.0

            seed_runs.append({
                "seed": seed,
                "useful_pairs": metrics["useful_pairs"],
                "attempted": metrics["attempted"],
                "halted": metrics["halted"],
                "qpu_yield_pct": yield_qpu_pct,
                "deficit_surplus": qpu_economy["useful_pairs_deficit_surplus"],
                "inference_latency_ms": metrics["avg_classical_latency_s"] * 1000.0,
                "throughput_pairs_per_s": throughput["throughput_pairs_per_s"],
                "qpu_cycles_saved": qpu_economy["qpu_cycles_saved"],
                "qpu_cycles_saved_pct": qpu_economy["qpu_cycles_saved_pct"],
                "total_energy_j": energy["total_energy_j"],
                "energy_saved_pct": energy["energy_saved_pct"],
            })

        if skipped or not seed_runs:
            continue

        per_model_seed_results[name] = seed_runs

        def _agg(key):
            return _mean_std([r[key] for r in seed_runs])

        yield_mean, yield_std = _agg("qpu_yield_pct")
        deficit_mean, deficit_std = _agg("deficit_surplus")
        latency_mean, latency_std = _agg("inference_latency_ms")
        throughput_mean, throughput_std = _agg("throughput_pairs_per_s")
        cycles_saved_pct_mean, cycles_saved_pct_std = _agg("qpu_cycles_saved_pct")
        energy_mean, energy_std = _agg("total_energy_j")
        energy_saved_pct_mean, energy_saved_pct_std = _agg("energy_saved_pct")
        useful_mean, useful_std = _agg("useful_pairs")

        rows.append({
            "Model": name,
            "N Seeds": len(seed_runs),
            "Useful Pairs": f"{useful_mean:.1f} +/- {useful_std:.1f}",
            "QPU Yield (%)": f"{yield_mean:.2f} +/- {yield_std:.2f}",
            "SKR Deficit/Surplus": f"{deficit_mean:+.1f} +/- {deficit_std:.1f}",
            "Throughput (pairs/s)": f"{throughput_mean:.2f} +/- {throughput_std:.2f}",
            "QPU Cycles Saved (%)": f"{cycles_saved_pct_mean:.2f} +/- {cycles_saved_pct_std:.2f}",
            "Energy (J)": f"{energy_mean:.6f} +/- {energy_std:.6f}",
            "Energy Saved (%)": f"{energy_saved_pct_mean:+.2f} +/- {energy_saved_pct_std:.2f}",
            "Inference Latency (ms)": f"{latency_mean:.4f} +/- {latency_std:.4f}",
            # Raw means, kept alongside the formatted strings above for
            # `build_decision_matrix` (which needs numeric values).
            "_qpu_yield_pct": yield_mean,
            "_throughput_pairs_per_s": throughput_mean,
            "_qpu_cycles_saved_pct": cycles_saved_pct_mean,
            "_energy_saved_pct": energy_saved_pct_mean,
            "_inference_latency_ms": latency_mean,
        })

        print(f"  -> QPU Yield (mean) = {yield_mean:.2f}% | Throughput (mean) = {throughput_mean:.2f} pairs/s | "
              f"QPU cycles saved (mean) = {cycles_saved_pct_mean:.2f}% | "
              f"Energy saved (mean) = {energy_saved_pct_mean:+.2f}% | "
              f"Latency (mean) = {latency_mean:.4f} ms\n")

    results_df = pd.DataFrame(rows, columns=[
        "Model", "N Seeds", "Useful Pairs", "QPU Yield (%)", "SKR Deficit/Surplus",
        "Throughput (pairs/s)", "QPU Cycles Saved (%)", "Energy (J)", "Energy Saved (%)",
        "Inference Latency (ms)",
    ])

    decision_rows = [{
        "Model": r["Model"],
        "qpu_yield_pct": r["_qpu_yield_pct"],
        "throughput_pairs_per_s": r["_throughput_pairs_per_s"],
        "qpu_cycles_saved_pct": r["_qpu_cycles_saved_pct"],
        "energy_saved_pct": r["_energy_saved_pct"],
        "inference_latency_ms": r["_inference_latency_ms"],
    } for r in rows]
    decision_matrix_df = build_decision_matrix(decision_rows, comparison_cfg.decision_weights, model_key="Model")

    return results_df, baseline_metrics, decision_matrix_df, per_model_seed_results


### 1.12 `qrepeater_twin/cli.py` -- `main()` and the command-line entry point


In [ ]:
%%writefile qrepeater_twin/cli.py
"""
Command-line entry point.

Previously, `main()` lived inside the notebook and could only be run cell
by cell, inside a Jupyter kernel. Here it is an ordinary, importable
library function (`from qrepeater_twin.cli import main`) and is also
runnable via `python -m qrepeater_twin.cli` from a plain terminal --
no notebook required.
"""

from __future__ import annotations

import argparse

import numpy as np
import torch

from .channel_simulator import WDMChannelSimulator
from .config import (
    BaselineConfig,
    ComparisonConfig,
    EnergyConfig,
    QuantumConfig,
    SimConfig,
    SweepConfig,
    TrainConfig,
)
from .pareto_sweep import run_pareto_sweep
from .model_comparison import run_model_comparison


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def main(sim_cfg: SimConfig = None, train_cfg: TrainConfig = None,
          quantum_cfg: QuantumConfig = None, sweep_cfg: SweepConfig = None,
          device: torch.device = None, base_seed: int = 42,
          run_baseline_comparison: bool = False,
          baseline_cfg: BaselineConfig = None, energy_cfg: EnergyConfig = None,
          comparison_cfg: ComparisonConfig = None):
    """
    Full pipeline of the Quantum Repeater Digital Twin (v3, decomposed).

    1. Generates and preprocesses the synthetic dataset (WDMChannelSimulator).
    2. Moves tensors to the selected device (CPU/GPU).
    3. Runs the (multi-seed) Pareto Frontier sweep over lambda_penalty,
       including the one-time computation of the blind/reactive baseline.
    4. Prints the consolidated metrics table.
    5. (Optional, `run_baseline_comparison=True`) Runs the cross-architecture
       baseline comparison (LSTM+MSE, Random Forest, XGBoost, Transformer)
       with throughput / QPU economy / energy accounting and prints the
       ranked decision matrix.
    """
    sim_cfg = sim_cfg or SimConfig()
    train_cfg = train_cfg or TrainConfig()
    quantum_cfg = quantum_cfg or QuantumConfig()
    sweep_cfg = sweep_cfg or SweepConfig()
    baseline_cfg = baseline_cfg or BaselineConfig()
    energy_cfg = energy_cfg or EnergyConfig()
    comparison_cfg = comparison_cfg or ComparisonConfig()
    device = device or get_device()

    torch.manual_seed(base_seed)
    np.random.seed(base_seed)

    print("=" * 88)
    print(" QUANTUM REPEATER DIGITAL TWIN -- PARETO FRONTIER (CS_MSELoss) ".center(88, "="))
    print("=" * 88)
    print(f"\nDevice: {device}")

    # -----------------------------------------------------------------
    # 1) Data generation and preprocessing
    # -----------------------------------------------------------------
    print("\n[1/3] Generating synthetic dataset (Ornstein-Uhlenbeck) ...")
    wdm_sim = WDMChannelSimulator(n_steps=sim_cfg.n_steps, dt=sim_cfg.dt, seed=sim_cfg.seed)
    df = wdm_sim.generate_dataset()
    X_train, y_train, X_test, y_test, feat_scaler = wdm_sim.preprocess(
        df, window_size=sim_cfg.window_size, test_size=sim_cfg.test_size,
    )
    print(f"    Total samples: {len(df)} | Training windows: {len(X_train)} | Test windows: {len(X_test)}")
    print(f"    Fraction of true fidelity below threshold {train_cfg.threshold}: "
          f"{(df['fidelity'] < train_cfg.threshold).mean() * 100:.1f}%")

    # -----------------------------------------------------------------
    # 2) Device handling: moves tensors to the selected device
    # -----------------------------------------------------------------
    X_train, y_train = X_train.to(device), y_train.to(device)
    X_test, y_test = X_test.to(device), y_test.to(device)

    # -----------------------------------------------------------------
    # 3) Pareto Frontier sweep over lambda_penalty (multi-seed averaging)
    # -----------------------------------------------------------------
    print(f"\n[2/3] Running the Pareto Frontier for lambda_penalty = "
          f"{sweep_cfg.lambda_values} with {len(sweep_cfg.seeds)} seeds/point "
          f"({sweep_cfg.seeds}) ...\n")

    results_df, baseline_metrics, per_seed_results = run_pareto_sweep(
        sweep_cfg.lambda_values, X_train, y_train, X_test, y_test, device=device,
        threshold=train_cfg.threshold, epochs=train_cfg.epochs, lr=train_cfg.lr,
        hidden_size=train_cfg.hidden_size,
        T1=quantum_cfg.T1, T2=quantum_cfg.T2, depol_prob=quantum_cfg.depol_prob,
        shots=quantum_cfg.shots, quantum_seed=quantum_cfg.seed,
        lambda_fn=train_cfg.lambda_fn, discard_penalty_weight=train_cfg.discard_penalty_weight,
        max_discard_rate=train_cfg.max_discard_rate, seeds=sweep_cfg.seeds,
    )

    # -----------------------------------------------------------------
    # Final report
    # -----------------------------------------------------------------
    print("[3/3] Consolidating results ...\n")
    print("=" * 88)
    print(" BASELINE (Blind/Reactive Purification) ".center(88, "="))
    print("=" * 88)
    print(f"  Total cycles evaluated    : {baseline_metrics['total_steps']}")
    print(f"  Purification attempts     : {baseline_metrics['attempted']} (unconditional admission)")
    print(f"  Useful pairs obtained     : {baseline_metrics['useful_pairs']}")
    print(f"  Forced classical latency  : {baseline_metrics['avg_classical_latency_s']*1000:.4f} ms "
          f"(neural network never invoked)")

    print("\n" + "=" * 88)
    print(" PARETO FRONTIER: CS_MSELoss(lambda_penalty) -- mean +/- std (multi-seed) "
          .center(88, "="))
    print("=" * 88)
    print(results_df.to_string(index=False))
    print("=" * 88)

    comparison_results = None
    if not run_baseline_comparison:
        return results_df, baseline_metrics, per_seed_results

    # -----------------------------------------------------------------
    # 4) (Optional) Cross-architecture baseline comparison
    # -----------------------------------------------------------------
    print("\n[4/4] Running cross-architecture baseline comparison "
          f"(LSTM+MSE, Random Forest, XGBoost, Transformer) with "
          f"{len(comparison_cfg.seeds)} seeds/model ...\n")

    comp_results_df, comp_baseline_metrics, decision_matrix_df, per_model_seed_results = run_model_comparison(
        X_train, y_train, X_test, y_test, device=device,
        train_cfg=train_cfg, quantum_cfg=quantum_cfg, baseline_cfg=baseline_cfg,
        energy_cfg=energy_cfg, comparison_cfg=comparison_cfg,
    )

    print("\n" + "=" * 88)
    print(" BASELINE COMPARISON: throughput / QPU economy / energy (mean +/- std) "
          .center(88, "="))
    print("=" * 88)
    print(comp_results_df.to_string(index=False))

    print("\n" + "=" * 88)
    print(" DECISION MATRIX (weighted, higher Decision Score = better; Rank 1 = recommended) "
          .center(88, "="))
    print("=" * 88)
    print(decision_matrix_df.to_string(index=False))
    print("=" * 88)

    comparison_results = (comp_results_df, comp_baseline_metrics, decision_matrix_df, per_model_seed_results)
    return results_df, baseline_metrics, per_seed_results, comparison_results


def build_arg_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description="Quantum Repeater Digital Twin -- Pareto Frontier")
    parser.add_argument("--epochs", type=int, default=150, help="Training epochs per round (seed x lambda).")
    parser.add_argument("--lr", type=float, default=0.012, help="Adam learning rate.")
    parser.add_argument("--hidden-size", type=int, default=16, help="EdgeLSTM hidden state size.")
    parser.add_argument("--seeds", type=int, nargs="+", default=[42, 43, 44, 45, 46],
                         help="Seeds used in multi-seed averaging (one training round per seed x lambda).")
    parser.add_argument("--lambda-values", type=float, nargs="+", default=[1.0, 2.0, 5.0, 10.0, 20.0, 50.0],
                         help="lambda_penalty values swept over the Pareto Frontier.")
    parser.add_argument("--compare-baselines", action="store_true",
                         help="Also run the cross-architecture baseline comparison "
                              "(LSTM+MSE, Random Forest, XGBoost, Transformer) with "
                              "throughput/QPU-economy/energy metrics and the decision matrix.")
    parser.add_argument("--no-xgboost", action="store_true",
                         help="Skip the XGBoost baseline even if the 'xgboost' package is installed.")
    parser.add_argument("--representative-lambda", type=float, default=10.0,
                         help="lambda_penalty used for the EdgeLSTM+CS-MSE row in --compare-baselines.")
    return parser


if __name__ == "__main__":
    args = build_arg_parser().parse_args()
    main(
        train_cfg=TrainConfig(epochs=args.epochs, lr=args.lr, hidden_size=args.hidden_size),
        sweep_cfg=SweepConfig(lambda_values=args.lambda_values, seeds=args.seeds),
        run_baseline_comparison=args.compare_baselines,
        comparison_cfg=ComparisonConfig(
            representative_lambda=args.representative_lambda,
            seeds=args.seeds,
            include_xgboost=not args.no_xgboost,
        ),
    )


### 1.13 Tests (`tests/`)

Minimal `pytest` suite covering `WDMChannelSimulator`, `EdgeLSTM`,
`CS_MSELoss`, `train_edge_lstm`, `InferenceTimer`, the baseline predictors
(`train_lstm_mse`, Random Forest, XGBoost, `TinyTransformer`), and the
`metrics.py` functions (throughput, QPU economy, energy, decision matrix).


In [ ]:
%%writefile tests/__init__.py


In [ ]:
%%writefile tests/test_channel_simulator.py
import numpy as np

from qrepeater_twin import WDMChannelSimulator


def test_generate_dataset_shape_and_bounds():
    sim = WDMChannelSimulator(n_steps=500, dt=0.01, seed=1)
    df = sim.generate_dataset()

    assert len(df) == 500
    assert set(df.columns) == {"phase_deviation", "temp_gradient", "fidelity"}
    assert (df["phase_deviation"] >= 0).all()
    assert (df["temp_gradient"] >= 0).all()
    assert (df["fidelity"] >= 0).all() and (df["fidelity"] <= 1).all()


def test_generate_dataset_is_deterministic_given_seed():
    df_a = WDMChannelSimulator(n_steps=200, seed=7).generate_dataset()
    df_b = WDMChannelSimulator(n_steps=200, seed=7).generate_dataset()
    assert np.allclose(df_a.values, df_b.values)


def test_preprocess_shapes():
    sim = WDMChannelSimulator(n_steps=500, dt=0.01, seed=1)
    df = sim.generate_dataset()
    X_train, y_train, X_test, y_test, scaler = sim.preprocess(df, window_size=20, test_size=0.2)

    n_windows = len(df) - 20
    expected_train = int(n_windows * 0.8)
    expected_test = n_windows - expected_train

    assert X_train.shape == (expected_train, 20, 2)
    assert y_train.shape == (expected_train, 1)
    assert X_test.shape == (expected_test, 20, 2)
    assert y_test.shape == (expected_test, 1)


In [ ]:
%%writefile tests/test_models_and_timing.py
import torch

from qrepeater_twin import CS_MSELoss, EdgeLSTM, InferenceTimer, train_edge_lstm


def test_edge_lstm_output_shape_and_range():
    torch.manual_seed(0)
    model = EdgeLSTM(input_size=2, hidden_size=8, num_layers=1)
    x = torch.rand(4, 20, 2)
    out = model(x)
    assert out.shape == (4, 1)
    assert torch.all(out >= 0.0) and torch.all(out <= 1.0)


def test_cs_mse_loss_penalizes_false_positive_more_than_false_negative():
    threshold = 0.65
    loss = CS_MSELoss(threshold=threshold, lambda_penalty=10.0, lambda_fn=2.0,
                       discard_penalty_weight=0.0, max_discard_rate=1.0)

    # False positive: ground truth below threshold, prediction above.
    y_true_fp = torch.tensor([[0.5]])
    y_pred_fp = torch.tensor([[0.9]])

    # False negative: ground truth above threshold, prediction below (same absolute error).
    y_true_fn = torch.tensor([[0.9]])
    y_pred_fn = torch.tensor([[0.5]])

    loss_fp = loss(y_pred_fp, y_true_fp).item()
    loss_fn = loss(y_pred_fn, y_true_fn).item()

    assert loss_fp > loss_fn


def test_train_edge_lstm_reduces_loss():
    torch.manual_seed(0)
    X = torch.rand(16, 20, 2)
    y = torch.rand(16, 1)

    model = EdgeLSTM(input_size=2, hidden_size=8, num_layers=1)
    criterion = CS_MSELoss()
    with torch.no_grad():
        loss_before = criterion(model(X), y).item()

    model = train_edge_lstm(model, X, y, epochs=50, lr=1e-2, seed=0)

    with torch.no_grad():
        loss_after = criterion(model(X), y).item()

    assert loss_after < loss_before


def test_inference_timer_cpu_returns_nonnegative_elapsed():
    device = torch.device("cpu")
    model = EdgeLSTM(input_size=2, hidden_size=8, num_layers=1)
    x = torch.rand(1, 20, 2)

    with InferenceTimer(device) as timer:
        _ = model(x)

    assert timer.elapsed_s >= 0.0
    assert timer.use_cuda_events is False


In [ ]:
%%writefile tests/test_baselines.py
import pytest
import torch

from qrepeater_twin import EdgeLSTM, TinyTransformer
from qrepeater_twin.baselines import (
    RandomForestFidelityModel,
    train_lstm_mse,
    train_random_forest,
    train_transformer,
    train_xgboost,
)


def test_train_lstm_mse_reduces_loss():
    torch.manual_seed(0)
    X = torch.rand(16, 20, 2)
    y = torch.rand(16, 1)

    model = EdgeLSTM(input_size=2, hidden_size=8, num_layers=1)
    criterion = torch.nn.MSELoss()
    with torch.no_grad():
        loss_before = criterion(model(X), y).item()

    model = train_lstm_mse(model, X, y, epochs=50, lr=1e-2, seed=0)

    with torch.no_grad():
        loss_after = criterion(model(X), y).item()

    assert loss_after < loss_before


def test_random_forest_fits_and_predicts_in_unit_range():
    torch.manual_seed(0)
    X_train = torch.rand(32, 20, 2)
    y_train = torch.rand(32, 1)

    rf_model = RandomForestFidelityModel(n_estimators=20, max_depth=4, seed=0)
    rf_model.fit(X_train, y_train)
    orchestrator_model = rf_model.as_orchestrator_model()

    x_sample = torch.rand(1, 20, 2)
    orchestrator_model.eval()
    pred = orchestrator_model(x_sample)

    assert pred.shape == (1, 1)
    assert torch.all(pred >= 0.0) and torch.all(pred <= 1.0)


def test_train_random_forest_convenience_function():
    torch.manual_seed(0)
    X_train = torch.rand(32, 20, 2)
    y_train = torch.rand(32, 1)

    model = train_random_forest(X_train, y_train, n_estimators=10, max_depth=3, seed=0)
    pred = model(torch.rand(1, 20, 2))
    assert pred.shape == (1, 1)


def test_train_xgboost_skips_cleanly_when_not_installed():
    pytest.importorskip("xgboost", reason="xgboost is an optional dependency for this baseline")
    torch.manual_seed(0)
    X_train = torch.rand(32, 20, 2)
    y_train = torch.rand(32, 1)
    model = train_xgboost(X_train, y_train, n_estimators=10, max_depth=3, seed=0)
    pred = model(torch.rand(1, 20, 2))
    assert pred.shape == (1, 1)


def test_tiny_transformer_output_shape_and_range():
    torch.manual_seed(0)
    model = TinyTransformer(input_size=2, d_model=8, nhead=2, num_layers=1, dim_feedforward=16)
    x = torch.rand(4, 20, 2)
    out = model(x)
    assert out.shape == (4, 1)
    assert torch.all(out >= 0.0) and torch.all(out <= 1.0)


def test_train_transformer_reduces_loss():
    torch.manual_seed(0)
    X = torch.rand(16, 20, 2)
    y = torch.rand(16, 1)

    model = TinyTransformer(input_size=2, d_model=8, nhead=2, num_layers=1, dim_feedforward=16)
    criterion = torch.nn.MSELoss()
    with torch.no_grad():
        loss_before = criterion(model(X), y).item()

    model = train_transformer(model, X, y, epochs=60, lr=5e-3, seed=0)

    with torch.no_grad():
        loss_after = criterion(model(X), y).item()

    assert loss_after < loss_before


In [ ]:
%%writefile tests/test_metrics.py
import pytest

from qrepeater_twin.config import EnergyConfig
from qrepeater_twin.metrics import (
    build_decision_matrix,
    compute_energy_report,
    compute_qpu_economy,
    compute_throughput,
)


def _baseline_metrics():
    return {"total_steps": 780, "useful_pairs": 300, "halted": 0,
            "attempted": 780, "avg_classical_latency_s": 0.0}


def _intelligent_metrics():
    return {"total_steps": 780, "useful_pairs": 280, "halted": 300,
            "attempted": 480, "avg_classical_latency_s": 0.00012}


def test_compute_throughput_positive_and_bounded_by_blind():
    baseline = compute_throughput(_baseline_metrics(), cycle_time_s=1e-3)
    intelligent = compute_throughput(_intelligent_metrics(), cycle_time_s=1e-3)

    assert baseline["throughput_pairs_per_s"] > 0
    assert intelligent["throughput_pairs_per_s"] > 0
    # The predictive controller adds classical latency on every cycle
    # (blind never pays it), so for equal cycle_time_s its total_time_s
    # is >= the blind baseline's.
    assert intelligent["total_time_s"] >= baseline["total_time_s"]


def test_compute_qpu_economy_reports_positive_savings_when_halting():
    economy = compute_qpu_economy(_intelligent_metrics(), _baseline_metrics(), shots_per_attempt=512)

    assert economy["qpu_cycles_saved"] == 300
    assert economy["qpu_cycles_saved_pct"] == pytest.approx(300 / 780 * 100.0)
    assert economy["qpu_shots_saved"] == 300 * 512
    assert economy["useful_pairs_deficit_surplus"] == -20


def test_compute_qpu_economy_without_shots_leaves_shots_saved_none():
    economy = compute_qpu_economy(_intelligent_metrics(), _baseline_metrics())
    assert economy["qpu_shots_saved"] is None


def test_compute_energy_report_zero_cycles_saved_means_equal_quantum_energy():
    # A "predictor" that never halts (attempted == baseline attempted)
    # must show identical *quantum*-side energy to the baseline; only the
    # classical inference term can differ.
    always_purify = dict(_intelligent_metrics())
    always_purify["attempted"] = 780
    always_purify["halted"] = 0

    report = compute_energy_report(always_purify, _baseline_metrics(), shots=512)
    baseline_quantum_only = compute_energy_report(_baseline_metrics(), _baseline_metrics(), shots=512)
    assert report["quantum_energy_j"] == pytest.approx(baseline_quantum_only["quantum_energy_j"])


def test_compute_energy_report_custom_energy_config_scales_linearly():
    cfg_low = EnergyConfig(joules_per_1q_gate=1e-9, joules_per_2q_gate=1e-9, joules_per_shot_overhead=0.0,
                            classical_inference_power_w=0.0, classical_idle_power_w=0.0)
    cfg_high = EnergyConfig(joules_per_1q_gate=2e-9, joules_per_2q_gate=2e-9, joules_per_shot_overhead=0.0,
                             classical_inference_power_w=0.0, classical_idle_power_w=0.0)

    metrics = _intelligent_metrics()
    baseline = _baseline_metrics()

    report_low = compute_energy_report(metrics, baseline, shots=512, energy_cfg=cfg_low)
    report_high = compute_energy_report(metrics, baseline, shots=512, energy_cfg=cfg_high)

    assert report_high["quantum_energy_j"] == pytest.approx(2 * report_low["quantum_energy_j"])


def test_build_decision_matrix_ranks_dominant_model_first():
    rows = [
        {
            "Model": "Blind",
            "qpu_yield_pct": 38.46,
            "throughput_pairs_per_s": 384.6,
            "qpu_cycles_saved_pct": 0.0,
            "energy_saved_pct": 0.0,
            "inference_latency_ms": 0.0,
        },
        {
            "Model": "Dominant",
            "qpu_yield_pct": 90.0,
            "throughput_pairs_per_s": 500.0,
            "qpu_cycles_saved_pct": 50.0,
            "energy_saved_pct": 40.0,
            "inference_latency_ms": 0.01,
        },
    ]
    weights = {
        "qpu_yield_pct": 0.25,
        "throughput_pairs_per_s": 0.20,
        "qpu_cycles_saved_pct": 0.20,
        "energy_saved_pct": 0.20,
        "inference_latency_ms": 0.15,
    }

    dm = build_decision_matrix(rows, weights)

    assert dm.iloc[0]["Model"] == "Dominant"
    assert dm.iloc[0]["Rank"] == 1
    assert dm.iloc[0]["Decision Score"] > dm.iloc[1]["Decision Score"]


def test_build_decision_matrix_missing_criterion_raises():
    rows = [{"Model": "A", "qpu_yield_pct": 50.0}]
    weights = {"qpu_yield_pct": 0.5, "throughput_pairs_per_s": 0.5}
    with pytest.raises(KeyError):
        build_decision_matrix(rows, weights)


def test_build_decision_matrix_empty_rows_returns_empty_frame():
    dm = build_decision_matrix([], {"qpu_yield_pct": 1.0})
    assert dm.empty


### 1.14 (Optional) run the test suite

Validates the decomposition (including the new baselines and metrics) before running the full pipeline.


In [ ]:
!python -m pytest tests/ -q


## 2. Import and device selection (CPU/GPU)

With the package materialized on disk, the rest of the notebook simply
imports it -- no business-logic class or function is redefined here.

In [ ]:
import torch

from qrepeater_twin.cli import get_device, main
from qrepeater_twin.config import (
    QuantumConfig, SimConfig, SweepConfig, TrainConfig,
    BaselineConfig, EnergyConfig, ComparisonConfig,
)

DEVICE = get_device()
print(f"Selected PyTorch device: {DEVICE}")


## 3. Experiment configuration

The same hyperparameters as v2, plus `SweepConfig.seeds`: the list of
seeds used in the multi-seed averaging at every point of the Pareto
Frontier. Increasing `len(seeds)` further reduces the reported variance, at
the cost of more training time (the cost scales linearly:
`len(lambda_values) * len(seeds)` full `EdgeLSTM` training runs).

`baseline_cfg`, `energy_cfg`, and `comparison_cfg` configure the new
cross-architecture baseline comparison (section 6 below): predictor
hyperparameters (LSTM+MSE / Random Forest / XGBoost / Transformer), the
illustrative energy-accounting coefficients, and the seeds / representative
`lambda_penalty` / decision-matrix weights used when ranking models.


In [ ]:
sim_cfg = SimConfig(n_steps=4000, dt=0.01, seed=42, window_size=20, test_size=0.2)

train_cfg = TrainConfig(
    hidden_size=16, epochs=150, lr=0.012, threshold=0.65,
    lambda_fn=4.0, discard_penalty_weight=10.0, max_discard_rate=0.60,
)

quantum_cfg = QuantumConfig(
    T1=50e-6, T2=30e-6, depol_prob=0.01, shots=512, seed=7, success_rate_cutoff=0.5,
)

sweep_cfg = SweepConfig(
    lambda_values=[1.0, 2.0, 5.0, 10.0, 20.0, 50.0],
    seeds=[42, 43, 44, 45, 46],  # multi-seed averaging: 5 independent rounds per lambda
)

# --- New in v3: baseline comparison configuration (see section 6) ---
baseline_cfg = BaselineConfig()      # LSTM+MSE / Random Forest / XGBoost / Transformer hyperparameters
energy_cfg = EnergyConfig()          # illustrative Joules-accounting coefficients
comparison_cfg = ComparisonConfig(
    representative_lambda=10.0,      # EdgeLSTM+CS-MSE row in the comparison uses this lambda
    seeds=[42, 43, 44, 45, 46],
    include_xgboost=True,            # set False (or leave xgboost uninstalled) to skip that row
    cycle_time_s=1e-3,
)


## 4. Execution

Calls `main()` from the decomposed package: generates the synthetic
dataset, runs the blind/reactive baseline exactly once, and sweeps the
Pareto Frontier over `lambda_penalty` with multi-seed averaging, printing
the final table (mean ± standard deviation per metric). With
`run_baseline_comparison=True`, it additionally runs the cross-architecture
baseline comparison (section 6) and prints the ranked decision matrix.


In [ ]:
results_df, baseline_metrics, per_seed_results, comparison_results = main(
    sim_cfg=sim_cfg, train_cfg=train_cfg, quantum_cfg=quantum_cfg,
    sweep_cfg=sweep_cfg, device=DEVICE, base_seed=42,
    run_baseline_comparison=True,
    baseline_cfg=baseline_cfg, energy_cfg=energy_cfg, comparison_cfg=comparison_cfg,
)


## 5. Per-seed audit (optional)

`per_seed_results` preserves the raw metrics of each individual round
(`lambda -> [{seed, halted, attempted, useful_pairs, yield_qpu_pct, ...}, ...]`),
allowing direct inspection of the spread across seeds -- for example, to
confirm that no single seed is dominating the mean in an anomalous way at
some lambda.

In [ ]:
import pandas as pd

lam_to_inspect = sweep_cfg.lambda_values[-1]  # e.g., the most conservative lambda in the sweep
pd.DataFrame(per_seed_results[lam_to_inspect])


## 6. Baseline comparison: throughput, QPU economy, energy, decision matrix

`comparison_results` (populated because `run_baseline_comparison=True` in
section 4) unpacks into:

- `comp_results_df` : one row per model (`EdgeLSTM+CS-MSE` at
  `comparison_cfg.representative_lambda`, `LSTM+MSE`, `RandomForest`,
  `XGBoost`, `Transformer`), mean ± std across `comparison_cfg.seeds` --
  useful pairs, QPU yield, throughput (pairs/s), QPU cycles saved (%),
  energy (J) and energy saved (%), inference latency (ms).
- `comp_baseline_metrics` : the blind/reactive baseline metrics, on the
  same test set (identical to `baseline_metrics` from the Pareto sweep).
- `decision_matrix_df` : `metrics.build_decision_matrix` output -- every
  weighted criterion normalized to [0, 1] (direction-aware: inference
  latency is a cost, so lower raw values score higher), a weighted
  `Decision Score`, and `Rank` (1 = recommended model given
  `comparison_cfg.decision_weights`).
- `per_model_seed_results` : raw per-seed metrics per model, for audit.


In [ ]:
comp_results_df, comp_baseline_metrics, decision_matrix_df, per_model_seed_results = comparison_results
comp_results_df


### 6.1 Decision matrix (ranked)


In [ ]:
decision_matrix_df


### 6.2 Per-seed audit for one model (optional)

Same idea as section 5, but for one of the baseline-comparison models --
inspect the spread across seeds before trusting the mean.


In [ ]:
model_to_inspect = "EdgeLSTM+CS-MSE"  # try "LSTM+MSE", "RandomForest", "XGBoost", "Transformer"
pd.DataFrame(per_model_seed_results[model_to_inspect])
